## 1. Mount Google Drive

Google Drive is mounted to provide persistent storage for the project files and intermediate results.

Using persistent storage prevents the loss of large reference datasets and computational outputs when the Colab runtime is disconnected or restarted.

In [5]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## 2. Import Python libraries

The following Python libraries are used throughout the pipeline.

- `pandas` is used for tabular GWAS summary-statistics processing.
- `numpy` is used for numerical operations.
- `pathlib` is used for platform-independent file and directory handling.
- `gzip` and `zipfile` are used when working with compressed GWAS files.
- `json` is used to save structured QC and pipeline reports.
- `subprocess` is used to execute PLINK2 commands from Python.
- `os` is used for selected operating-system-level operations.

Only libraries required by the pipeline are imported.

In [2]:
import argparse
import os
import subprocess
import sys
import gzip
import warnings
from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

## 3. Record the computational environment

Recording the software environment improves reproducibility.

The Python version and versions of the main Python libraries are recorded before analysis begins. PLINK2 is also checked separately because it is the primary command-line tool used for LD reference construction, chromosome-wise processing, LD clumping, and related genotype operations.

In [3]:
print(f"Python:       {sys.version.split()[0]}")
print("\nPackage versions:")
print(f"NumPy:        {np.__version__}")
print(f"Pandas:       {pd.__version__}")
print(f"Matplotlib:   {matplotlib.__version__}")

Python:       3.13.15

Package versions:
NumPy:        2.1.3
Pandas:       2.2.3
Matplotlib:   3.10.0


## 4. Install and verify PLINK2

PLINK2 is used for genotype-reference processing and LD-based analyses.

The executable is installed locally in the Colab runtime. The executable itself is not treated as a persistent project result because the Colab runtime can be recreated when necessary.

The version is recorded to document the computational environment used for the analysis.

In [4]:
!wget https://s3.amazonaws.com/plink2-assets/plink2_linux_x86_64_latest.zip
!unzip -o plink2_linux_x86_64_latest.zip -d plink2

--2026-10-04 09:08:01--  https://s3.amazonaws.com/plink2-assets/plink2_linux_x86_64_latest.zip
Resolving s3.amazonaws.com (s3.amazonaws.com)... 52.217.228.224, 16.15.244.60, 52.217.196.192, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|52.217.228.224|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10226431 (9.8M) [application/zip]
Saving to: ‘plink2_linux_x86_64_latest.zip’

plink2_linux_x86_64 100%[===================>]   9.75M  38.0MB/s    in 0.3s    

2026-10-04 09:08:02 (38.0 MB/s) - ‘plink2_linux_x86_64_latest.zip’ saved [10226431/10226431]

Archive:  plink2_linux_x86_64_latest.zip
  inflating: plink2/plink2           
  inflating: plink2/ldsc             
  inflating: plink2/mtag             
  inflating: plink2/vcf_subset       
  inflating: plink2/intel-simplified-software-license.txt  


In [5]:
PLINK = "/content/plink2/plink2"

result = subprocess.run(
    [PLINK, "--version"],
    capture_output=True,
    text=True
)

print(result.stdout)

PLINK v2.0.0-b.1-devLM 64-bit Intel (1 Oct 2026)



## 5. Create the project directory structure

A complete project directory is created on Google Drive before the analysis begins.

The project root is `PRS_T2D_E-A`.

The directory structure separates:

- raw GWAS input;
- persistent LD reference files;
- intermediate candidate SNP files;
- LD-clumping inputs and outputs;
- PRS weights; and
- computational results.

Creating the complete structure at the beginning makes the analysis easier to reproduce and prevents intermediate files from being scattered across Google Drive.

In [6]:
# Project root
PROJECT = Path("/content/drive/MyDrive/PRS_T2D_E-A")

# Main directories
GWAS_DIR = PROJECT / "GWAS"
REFERENCE_DIR = PROJECT / "reference"
RESULTS_DIR = PROJECT / "results"

# Reference subdirectory
EA_CHUNKS_DIR = REFERENCE_DIR / "EA_candidate_chunks"

# Results subdirectories
GWAS_QC_DIR = RESULTS_DIR / "gwas_qc"
HARMONIZED_DIR = RESULTS_DIR / "harmonized"
CANDIDATE_DIR = RESULTS_DIR / "candidate_snps"
LD_DIR = RESULTS_DIR / "ld_clumping"
WEIGHTS_DIR = RESULTS_DIR / "weights"

# LD subdirectories
THRESHOLD_CLUMPS_DIR = LD_DIR / "threshold_clumps"
NORMAL_CLUMPS_DIR = LD_DIR / "normal_clumps"
REFERENCE_CHUNKS_DIR = LD_DIR / "reference_chunks"
LD_LOG_DIR = LD_DIR / "logs"

# Create all directories
directories = [
    PROJECT,
    GWAS_DIR,
    REFERENCE_DIR,
    EA_CHUNKS_DIR,
    RESULTS_DIR,
    GWAS_QC_DIR,
    HARMONIZED_DIR,
    CANDIDATE_DIR,
    LD_DIR,
    THRESHOLD_CLUMPS_DIR,
    NORMAL_CLUMPS_DIR,
    REFERENCE_CHUNKS_DIR,
    LD_LOG_DIR,
    WEIGHTS_DIR,
]

# Threshold-specific weight directories
THRESHOLDS = {
    "P5e-08": 5e-8,
    "P1e-05": 1e-5,
    "P0.0001": 1e-4,
    "P0.001": 1e-3,
    "P0.01": 1e-2,
    "P0.05": 5e-2,
}

for directory in directories:
    directory.mkdir(parents=True, exist_ok=True)

for threshold_name in THRESHOLDS:
    (WEIGHTS_DIR / threshold_name).mkdir(
        parents=True,
        exist_ok=True
    )

print("Project directory structure created successfully.")
print(PROJECT)

Project directory structure created successfully.
/content/drive/MyDrive/PRS_T2D_E-A


## 6. Verify the project directory structure

The expected project directories are checked before proceeding.

This provides an early structural checkpoint and ensures that subsequent outputs have predefined locations.

In [7]:
for directory in directories:
    print(f"{directory}: {'OK' if directory.exists() else 'MISSING'}")

/content/drive/MyDrive/PRS_T2D_E-A: OK
/content/drive/MyDrive/PRS_T2D_E-A/GWAS: OK
/content/drive/MyDrive/PRS_T2D_E-A/reference: OK
/content/drive/MyDrive/PRS_T2D_E-A/reference/EA_candidate_chunks: OK
/content/drive/MyDrive/PRS_T2D_E-A/results: OK
/content/drive/MyDrive/PRS_T2D_E-A/results/gwas_qc: OK
/content/drive/MyDrive/PRS_T2D_E-A/results/harmonized: OK
/content/drive/MyDrive/PRS_T2D_E-A/results/candidate_snps: OK
/content/drive/MyDrive/PRS_T2D_E-A/results/ld_clumping: OK
/content/drive/MyDrive/PRS_T2D_E-A/results/ld_clumping/threshold_clumps: OK
/content/drive/MyDrive/PRS_T2D_E-A/results/ld_clumping/normal_clumps: OK
/content/drive/MyDrive/PRS_T2D_E-A/results/ld_clumping/reference_chunks: OK
/content/drive/MyDrive/PRS_T2D_E-A/results/ld_clumping/logs: OK
/content/drive/MyDrive/PRS_T2D_E-A/results/weights: OK


## 7. Add the T2D GWAS summary statistics

The T2D GWAS summary-statistics file has already been downloaded independently and is manually uploaded to the `GWAS` directory in Google Drive.

The computational pipeline does not download the GWAS automatically.

This separation ensures that the analysis notebook begins from a clearly identified input file and that the downloaded source data can be retained independently of the computational workflow.

### Input

The correct GWAS summary-statistics file for type 2 diabetes should be placed in:

`PRS_T2D_E-A/GWAS/`

After uploading the file, its exact filename should be recorded below.

In [8]:
GWAS_FILE = GWAS_DIR / "MVP_T2D_Phe_250_2_EUR_raw.txt.gz"

print(GWAS_FILE)
print("Exists:", GWAS_FILE.exists())

/content/drive/MyDrive/PRS_T2D_E-A/GWAS/MVP_T2D_Phe_250_2_EUR_raw.txt.gz
Exists: True


## 8. Verify the GWAS input file

Before performing any analysis, the uploaded file is checked for:

- existence;
- filename;
- file size; and
- readable structure.

This step confirms that the intended GWAS summary-statistics file is being used before computational processing begins.

In [9]:
if not GWAS_FILE.exists():
    raise FileNotFoundError(
        f"GWAS file not found: {GWAS_FILE}"
    )

print("GWAS file found.")
print("Path:", GWAS_FILE)
print("Size (MB):", round(GWAS_FILE.stat().st_size / 1024**2, 2))

GWAS file found.
Path: /content/drive/MyDrive/PRS_T2D_E-A/GWAS/MVP_T2D_Phe_250_2_EUR_raw.txt.gz
Size (MB): 554.76


## 9. Inspect the discovery GWAS

The discovery GWAS is inspected before quality control.

The purpose is to confirm that:

1. the file is readable;
2. the expected columns are present;
3. the variant-level structure is appropriate for PRS construction;
4. allele, frequency, effect-size, standard-error and P-value fields are available; and
5. the dataset corresponds to the intended T2D GWAS.

The GWAS identity and metadata should be checked against the original study documentation before downstream analysis proceeds.

In [10]:
# Preview the first five rows
gwas_preview = pd.read_csv(
    GWAS_FILE,
    sep="\t",
    compression="infer",
    nrows=5
)

display(gwas_preview)

print("\nColumns:")
print(gwas_preview.columns.tolist())

print("\nData types:")
print(gwas_preview.dtypes)
print("Number of columns:", len(gwas_preview.columns))
print("GWAS filename:")
print(GWAS_FILE.name)

,SNP_ID,chrom,pos,ref,alt,ea,af,num_samples,case_af,num_cases,control_af,num_controls,or,ci,pval,r2,q_pval,i2,direction
0,rs558358031,1,60879,G,A,A,0.9994,432648,0.9994,154194,0.9994,278454,0.9731,"0.6348,1.492",0.9003,0.3310,NaN,NaN,NaN
1,rs562476259,1,85941,T,C,C,0.9987,432648,0.9987,154194,0.9987,278454,1.1670,"0.883,1.543",0.2775,0.3228,NaN,NaN,NaN
2,rs567575968,1,133166,A,G,G,0.9998,432648,0.9998,154194,0.9998,278454,0.9692,"0.5685,1.652",0.9085,0.4974,NaN,NaN,NaN
3,rs573978839,1,268598,T,G,G,0.9995,432648,0.9995,154194,0.9995,278454,1.0190,"0.6642,1.564",0.9304,0.3631,NaN,NaN,NaN
4,rs531691745,1,284844,C,A,A,0.9980,432648,0.9980,154194,0.9980,278454,0.9005,"0.7179,1.13",0.3647,0.3389,NaN,NaN,NaN



Columns:
['SNP_ID', 'chrom', 'pos', 'ref', 'alt', 'ea', 'af', 'num_samples', 'case_af', 'num_cases', 'control_af', 'num_controls', 'or', 'ci', 'pval', 'r2', 'q_pval', 'i2', 'direction']

Data types:
SNP_ID           object
chrom             int64
pos               int64
ref              object
alt              object
ea               object
af              float64
num_samples       int64
case_af         float64
num_cases         int64
control_af      float64
num_controls      int64
or              float64
ci               object
pval            float64
r2              float64
q_pval          float64
i2              float64
direction       float64
dtype: object
Number of columns: 19
GWAS filename:
MVP_T2D_Phe_250_2_EUR_raw.txt.gz


### Discovery GWAS metadata

Trait: Type 2 diabetes mellitus

Study:
[insert study name]

Population:
[insert population]

Cases:
[insert number]

Controls:
[insert number]

Total participants:
[insert number]

Genome build:
[insert build]

Association method:
[insert method]

Source:
[insert source/database]

The metadata above were verified against the original GWAS documentation before analysis.

In [12]:
# DISCOVERY GWAS METADATA EXTRACTION
# Dataset: MVP PheCode 250.2 (Type 2 diabetes)
# Population: European ancestry
# Analysis: Phe_250_2.EUR.GIA

import gzip
import json
import pandas as pd

# 1. USER SETTINGS

GWAS_FILE = (
    "/content/drive/MyDrive/PRS_T2D_E-A/GWAS/"
    "MVP_T2D_Phe_250_2_EUR_raw.txt.gz"
)

OUTPUT_JSON = (
    "/content/drive/MyDrive/PRS_T2D_E-A/GWAS/"
    "MVP_T2D_Phe_250_2_EUR_metadata.json"
)

# 2. READ HEADER AND SMALL SAMPLE

with gzip.open(GWAS_FILE, "rt") as f:
    header = f.readline().strip().split("\t")

print("EUROPEAN T2D GWAS SUMMARY STATISTICS")

print("\nGWAS file:")
print(GWAS_FILE)

print("\nSummary-statistics columns:")
print(header)

gwas_preview = pd.read_csv(
    GWAS_FILE,
    sep="\t",
    compression="gzip",
    nrows=5
)

print("\nFirst 5 rows:")
display(gwas_preview)

# 3. DATA TYPES AND BASIC STRUCTURE

print("SUMMARY-STATISTICS STRUCTURE")

print("\nData types:")
print(gwas_preview.dtypes)

print("\nNumber of columns:")
print(len(header))

# 4. SAMPLE-SIZE INFORMATION


sample_info = pd.read_csv(
    GWAS_FILE,
    sep="\t",
    compression="gzip",
    usecols=[
        "num_samples",
        "num_cases",
        "num_controls"
    ]
)


print("SAMPLE-SIZE INFORMATION")

print(
    "\nMinimum variant-level N:",
    sample_info["num_samples"].min()
)

print(
    "Maximum variant-level N:",
    sample_info["num_samples"].max()
)

print(
    "Median variant-level N:",
    sample_info["num_samples"].median()
)

print(
    "Unique N values:",
    sample_info["num_samples"].nunique()
)

print(
    "\nUnique number of cases:",
    sample_info["num_cases"].unique()
)

print(
    "Unique number of controls:",
    sample_info["num_controls"].unique()
)

# 5. DISCOVERY GWAS METADATA

metadata = {

    "Trait": "Type 2 diabetes mellitus",

    "PheCode": "250.2",

    "Analysis": "Phe_250_2.EUR.GIA",

    "Population": (
        "MVP participants of European ancestry"
    ),

    "Dataset": (
        "Million Veteran Program (MVP) "
        "PheCode 250.2 European ancestry GWAS"
    ),

    "Cases": 154194,

    "Controls": 278454,

    "Total participants": 432648,

    "Genome build": "GRCh37 / hg19",

    "Association method": (
        "SAIGE mixed-model GWAS"
    ),

    "Effect measure": (
        "Odds ratio (OR)"
    ),

    "Effect allele": "ea",

    "Other allele": (
        "Allele not represented by the effect allele"
    ),

    "Allele frequency": "af",

    "P-value": "pval",

    "Variant identifier": "SNP_ID",

    "Chromosome": "chrom",

    "Position": "pos",

    "Source": (
        "Million Veteran Program (MVP), "
        "Phe_250_2.EUR.GIA"
    ),

    "Input file": (
        "MVP_T2D_Phe_250_2_EUR_raw.txt.gz"
    )
}

# 6. PRINT METADATA

print("DISCOVERY GWAS METADATA")

for key, value in metadata.items():

    print(f"\n{key}:")
    print(value)

# 7. SAVE METADATA

with open(OUTPUT_JSON, "w") as f:
    json.dump(metadata, f, indent=4)

print("\n" + "=" * 70)
print("Metadata saved to:")
print(OUTPUT_JSON)
print("=" * 70)

EUROPEAN T2D GWAS SUMMARY STATISTICS

GWAS file:
/content/drive/MyDrive/PRS_T2D_E-A/GWAS/MVP_T2D_Phe_250_2_EUR_raw.txt.gz

Summary-statistics columns:
['SNP_ID', 'chrom', 'pos', 'ref', 'alt', 'ea', 'af', 'num_samples', 'case_af', 'num_cases', 'control_af', 'num_controls', 'or', 'ci', 'pval', 'r2', 'q_pval', 'i2', 'direction']

First 5 rows:


,SNP_ID,chrom,pos,ref,alt,ea,af,num_samples,case_af,num_cases,control_af,num_controls,or,ci,pval,r2,q_pval,i2,direction
0,rs558358031,1,60879,G,A,A,0.9994,432648,0.9994,154194,0.9994,278454,0.9731,"0.6348,1.492",0.9003,0.3310,NaN,NaN,NaN
1,rs562476259,1,85941,T,C,C,0.9987,432648,0.9987,154194,0.9987,278454,1.1670,"0.883,1.543",0.2775,0.3228,NaN,NaN,NaN
2,rs567575968,1,133166,A,G,G,0.9998,432648,0.9998,154194,0.9998,278454,0.9692,"0.5685,1.652",0.9085,0.4974,NaN,NaN,NaN
3,rs573978839,1,268598,T,G,G,0.9995,432648,0.9995,154194,0.9995,278454,1.0190,"0.6642,1.564",0.9304,0.3631,NaN,NaN,NaN
4,rs531691745,1,284844,C,A,A,0.9980,432648,0.9980,154194,0.9980,278454,0.9005,"0.7179,1.13",0.3647,0.3389,NaN,NaN,NaN


SUMMARY-STATISTICS STRUCTURE

Data types:
SNP_ID           object
chrom             int64
pos               int64
ref              object
alt              object
ea               object
af              float64
num_samples       int64
case_af         float64
num_cases         int64
control_af      float64
num_controls      int64
or              float64
ci               object
pval            float64
r2              float64
q_pval          float64
i2              float64
direction       float64
dtype: object

Number of columns:
19
SAMPLE-SIZE INFORMATION

Minimum variant-level N: 432648
Maximum variant-level N: 432648
Median variant-level N: 432648.0
Unique N values: 1

Unique number of cases: [154194]
Unique number of controls: [278454]
DISCOVERY GWAS METADATA

Trait:
Type 2 diabetes mellitus

PheCode:
250.2

Analysis:
Phe_250_2.EUR.GIA

Population:
MVP participants of European ancestry

Dataset:
Million Veteran Program (MVP) PheCode 250.2 European ancestry GWAS

Cases:
154194

Controls

## 10. Quality control of the discovery GWAS

Quality control is performed to remove variants that cannot be reliably used for downstream PRS construction.

The QC procedure evaluates:

- missing required fields;
- invalid allele representations;
- invalid allele frequencies;
- invalid effect estimates;
- invalid standard errors;
- invalid P-values; and
- duplicate variant identifiers.

A structured QC report is saved so that every filtering decision can be documented.

In [13]:
# T2D GWAS SUMMARY STATISTICS — EUROPEAN ANCESTRY
# STANDARDIZATION + SE DERIVATION + QUALITY CONTROL
# Dataset: MVP PheCode 250.2 — European ancestry GIA

# Workflow:
# Raw EUR GWAS
#       ↓
# Standardization
#       ↓
# Derive BETA from OR
#       ↓
# Derive SE from 95% CI
#       ↓
# Inspect standardized data
#       ↓
# Main GWAS QC
#       ↓
# Verified QC output


# ============================================================
# 1. IMPORTS
# ============================================================

import gzip
import json
import time
import shutil
import re

import numpy as np
import pandas as pd

from pathlib import Path


# ============================================================
# 2. PATHS
# ============================================================

# Raw EUR GWAS input

GWAS_FILE = Path(
    "/content/drive/MyDrive/PRS_T2D_E-A/GWAS/"
    "MVP_T2D_Phe_250_2_EUR_raw.txt.gz"
)


# QC output directory
#
# IMPORTANT:
# The project structure is:
#
# PRS_T2D_E-A/
# ├── GWAS/
# ├── reference/
# └── results/
#     └── gwas_qc/
#
# Therefore, gwas_qc is NOT inside the GWAS folder.

GWAS_QC_DIR = Path(
    "/content/drive/MyDrive/PRS_T2D_E-A/"
    "results/gwas_qc/"
)


# Make sure the QC output directory exists

GWAS_QC_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Final QC output

QC_OUTPUT = (
    GWAS_QC_DIR /
    "MVP_T2D_Phe_250_2_EUR_QC.tsv.gz"
)


# QC report

QC_REPORT = (
    GWAS_QC_DIR /
    "MVP_T2D_Phe_250_2_EUR_QC_report.json"
)


# ------------------------------------------------------------
# Local temporary QC output
#
# Processing locally first avoids repeatedly writing
# large chunks directly to Google Drive.
# ------------------------------------------------------------

LOCAL_QC_OUTPUT = Path(
    "/content/"
    "MVP_T2D_Phe_250_2_EUR_QC.tsv.gz"
)


# ============================================================
# 3. CHECK INPUT FILE
# ============================================================

if not GWAS_FILE.exists():

    raise FileNotFoundError(
        f"GWAS file not found:\n{GWAS_FILE}"
    )


print(
    "T2D GWAS SUMMARY STATISTICS — EUROPEAN ANCESTRY"
)

print(
    "STANDARDIZATION + SE DERIVATION + QUALITY CONTROL"
)

print(
    f"\nInput file:\n{GWAS_FILE}"
)

print(
    f"\nInput compressed size: "
    f"{GWAS_FILE.stat().st_size / (1024**3):.2f} GB"
)


# ============================================================
# 4. RAW EUR GWAS COLUMNS
# ============================================================

# The raw EUR MVP file contains columns such as:
#
# SNP_ID
# chrom
# pos
# ref
# alt
# ea
# af
# num_samples
# num_cases
# num_controls
# or
# ci
# pval
#
# These will be converted to the standardized columns used
# throughout the PRS pipeline.


RAW_USECOLS = [
    "SNP_ID",
    "chrom",
    "pos",
    "ref",
    "alt",
    "ea",
    "af",
    "num_samples",
    "or",
    "ci",
    "pval",
]


# ============================================================
# 5. STANDARDIZED OUTPUT COLUMNS
# ============================================================

# These are the columns expected by the downstream PRS
# pipeline.
#
# SNP          = variant identifier
# CHR          = chromosome
# BP           = base-pair position
# A1           = effect allele
# A2           = other allele
# FREQ         = effect allele frequency
# BETA         = log(OR)
# SE           = standard error derived from 95% CI
# P            = association P-value
# N            = GWAS sample size
#
# SE_DERIVED   = provenance flag showing that SE was derived
#                from the reported confidence interval.


STANDARDIZED_COLUMNS = [
    "SNP",
    "CHR",
    "BP",
    "A1",
    "A2",
    "FREQ",
    "BETA",
    "SE",
    "P",
    "N",
    "SE_DERIVED",
]


valid_alleles = {
    "A",
    "C",
    "G",
    "T"
}


# ============================================================
# 6. SE DERIVATION FUNCTION
# ============================================================

# The EUR MVP dataset reports:
#
#     OR
#     95% confidence interval
#
# rather than a direct standard error.
#
# First:
#
#     BETA = ln(OR)
#
# Then, assuming the reported CI is a 95% Wald-type interval:
#
#     SE = [ln(CI_upper) - ln(CI_lower)] / (2 × 1.96)
#
# This is equivalent to:
#
#     SE = [ln(CI_upper) - ln(CI_lower)] / 3.92


def parse_ci(ci_value):
    """
    Extract lower and upper confidence limits from the
    CI string.

    Expected format:
        0.6348,1.492

    Also accepts surrounding parentheses.
    """

    if pd.isna(ci_value):

        return np.nan, np.nan


    ci_string = str(
        ci_value
    ).strip()


    match = re.match(
        r"^\s*\(?\s*"
        r"([0-9eE.+-]+)"
        r"\s*[,;]\s*"
        r"([0-9eE.+-]+)"
        r"\s*\)?\s*$",
        ci_string
    )


    if match is None:

        return np.nan, np.nan


    lower = pd.to_numeric(
        match.group(1),
        errors="coerce"
    )


    upper = pd.to_numeric(
        match.group(2),
        errors="coerce"
    )


    return lower, upper


# ============================================================
# 7. INITIALIZE QC STATISTICS
# ============================================================

stats = {

    "input_rows": 0,

    "qc_passed_rows": 0,

    "removed_missing": 0,

    "removed_invalid_snp": 0,

    "removed_invalid_alleles": 0,

    "removed_invalid_chr": 0,

    "removed_invalid_bp": 0,

    "removed_invalid_freq": 0,

    "removed_invalid_or": 0,

    "removed_invalid_ci": 0,

    "removed_invalid_beta": 0,

    "removed_invalid_se": 0,

    "removed_invalid_p": 0,

    "removed_invalid_n": 0,

    "removed_duplicate_snp": 0,

    "chunks_completed": 0,

    "se_derivation_formula": (
        "SE = [ln(CI_upper) - ln(CI_lower)] / "
        "(2 × 1.96)"
    ),

    "beta_derivation_formula": (
        "BETA = ln(OR)"
    ),

    "se_source": (
        "Derived from reported 95% confidence interval"
    ),

}


# ============================================================
# 8. INITIALIZE PROCESSING
# ============================================================

seen_snps = set()

start_time = time.time()

first_write = True

CHUNK_SIZE = 100_000


print(
    "\nChunk size:"
    f" {CHUNK_SIZE:,}"
)


print(
    f"Local QC output:\n{LOCAL_QC_OUTPUT}"
)


# ============================================================
# 9. READ RAW EUR GWAS IN CHUNKS
# ============================================================

reader = pd.read_csv(

    GWAS_FILE,
    sep="\t",
    compression="gzip",
    usecols=RAW_USECOLS,
    chunksize=CHUNK_SIZE,
    low_memory=False,
)


# ============================================================
# 10. PROCESS EACH CHUNK
# ============================================================

for chunk_number, chunk in enumerate(
    reader,
    start=1
):

    # --------------------------------------------------------
    # Count original rows
    # --------------------------------------------------------

    stats["input_rows"] += len(chunk)


    # ========================================================
    # 10A. STANDARDIZATION
    # ========================================================

    # This section converts the raw EUR MVP column names and
    # values into the standardized PRS pipeline format.
    #
    # The main QC filters have NOT started yet.


    # --------------------------------------------------------
    # SNP identifier
    #
    # SNP_ID → SNP
    # --------------------------------------------------------

    chunk["SNP"] = (
        chunk["SNP_ID"]
        .astype(str)
        .str.strip()
    )


    # --------------------------------------------------------
    # Chromosome
    #
    # Raw examples:
    #     chr1
    #     chr2
    #
    # Standardized:
    #     1
    #     2
    # --------------------------------------------------------

    chunk["CHR"] = (
        chunk["chrom"]
        .astype(str)
        .str.strip()
        .str.replace(
            r"^chr",
            "",
            regex=True,
            case=False
        )
    )


    # Base-pair position
    #
    # pos → BP

    chunk["BP"] = chunk["pos"]



    # Effect allele
    #
    # ea → A1


    chunk["A1"] = (
        chunk["ea"]
        .astype(str)
        .str.upper()
        .str.strip()
    )



    # Alleles
    #
    # ref / alt are used to determine the allele that is not
    # the effect allele.
    #
    # If:
    #     ea == ref
    #
    # then:
    #     A2 = alt
    #
    # Otherwise:
    #     A2 = ref
    #
    # This is done only after checking that EA is one of
    # REF or ALT.


    chunk["ref"] = (
        chunk["ref"]
        .astype(str)
        .str.upper()
        .str.strip()
    )


    chunk["alt"] = (
        chunk["alt"]
        .astype(str)
        .str.upper()
        .str.strip()
    )


    effect_is_ref = (
        chunk["A1"] == chunk["ref"]
    )


    effect_is_alt = (
        chunk["A1"] == chunk["alt"]
    )


    valid_effect_allele = (
        effect_is_ref
        |
        effect_is_alt
    )


    chunk["A2"] = np.where(

        effect_is_ref,

        chunk["alt"],

        np.where(
            effect_is_alt,
            chunk["ref"],
            np.nan
        )
    )


    # Effect allele frequency

    # af → FREQ

    chunk["FREQ"] = chunk["af"]


    # Sample size
    # num_samples → N

    chunk["N"] = chunk["num_samples"]



    # 10B. DERIVE BETA FROM ODDS RATIO

    # The raw EUR dataset reports OR rather than BETA.
    #
    # For a logistic GWAS:
    #
    #     BETA = ln(OR)
    #
    # Therefore the OR is converted to the natural-log scale
    # before entering the standardized PRS pipeline.


    chunk["OR"] = pd.to_numeric(
        chunk["or"],
        errors="coerce"
    )


    chunk["BETA"] = np.where(

        chunk["OR"] > 0,

        np.log(
            chunk["OR"]
        ),

        np.nan
    )


    # 10C. DERIVE SE FROM 95% CONFIDENCE INTERVAL

    # The reported CI is parsed into:
    #
    #     CI_LOWER
    #     CI_UPPER
    #
    # Then:
    #
    #     SE =
    #     [ln(CI_UPPER) - ln(CI_LOWER)]
    #     --------------------------------
    #              2 × 1.96
    #
    # A separate SE_DERIVED column is retained so that the
    # provenance of the SE is explicit.


    ci_values = (
        chunk["ci"]
        .apply(parse_ci)
    )


    chunk["CI_LOWER"] = (
        ci_values.str[0]
    )


    chunk["CI_UPPER"] = (
        ci_values.str[1]
    )


    # Derive SE

    valid_ci = (

        (chunk["CI_LOWER"] > 0)

        &

        (chunk["CI_UPPER"] > 0)

        &

        (
            chunk["CI_UPPER"]
            >
            chunk["CI_LOWER"]
        )
    )


    chunk["SE"] = np.where(

        valid_ci,

        (
            np.log(
                chunk["CI_UPPER"]
            )
            -
            np.log(
                chunk["CI_LOWER"]
            )
        )
        /
        (2 * 1.96),

        np.nan
    )


    # Explicit provenance column

    chunk["SE_DERIVED"] = (
        chunk["SE"].notna()
    )


    # P-value
    #
    # pval → P

    chunk["P"] = chunk["pval"]


    # 10D. STANDARDIZED DATASET PREVIEW

    # Print the first five standardized rows ONLY on the
    # first chunk.

    if chunk_number == 1:

        print(
            "STANDARDIZATION PREVIEW"
        )

        preview_columns = [

            "SNP",
            "CHR",
            "BP",
            "A1",
            "A2",
            "FREQ",
            "OR",
            "BETA",
            "CI_LOWER",
            "CI_UPPER",
            "SE",
            "SE_DERIVED",
            "P",
            "N",
        ]


        print(

            chunk[
                preview_columns
            ]
            .head(5)
            .to_string(
                index=False
            )
        )


        print(
            "\nSE DERIVATION CHECK"
        )


        print(
            "SE source: "
            "reported 95% confidence interval"
        )


        print(
            "Formula: "
            "SE = [ln(CI_upper) - ln(CI_lower)] "
            "/ (2 × 1.96)"
        )


        print(

            "\nNumber of SE values successfully derived "
            "in first chunk: "

            f"{chunk['SE'].notna().sum():,}"
        )


        print(

            "Number of SE_DERIVED = True: "

            f"{chunk['SE_DERIVED'].sum():,}"
        )


        print(
            "\nStandardization preview complete."
        )


        print(
            "Main QC begins below."
        )

    # 10E. MAIN GWAS QUALITY CONTROL STARTS HERE

    before = len(chunk)

    # Missing raw/standardized values

    required_for_qc = [

        "SNP",
        "CHR",
        "BP",
        "A1",
        "A2",
        "FREQ",
        "BETA",
        "SE",
        "P",
        "N",
    ]


    chunk = chunk.dropna(
        subset=required_for_qc
    )

    stats["removed_missing"] += (
        before
        -
        len(chunk)
    )


    # SNP IDENTIFIERS

    snp_mask = (

        chunk["SNP"].notna()

        &

        (chunk["SNP"] != "")

        &

        (
            chunk["SNP"].str.lower()
            != "nan"
        )
    )


    stats["removed_invalid_snp"] += (
        (~snp_mask).sum()
    )


    chunk = chunk[snp_mask]


    # ALLELES

    # Require:
    #   A1 and A2 ∈ {A,C,G,T}

    #   A1 != A2
    #   EA must have been successfully matched to REF/ALT

    allele_mask = (

        chunk["A1"].isin(
            valid_alleles
        )

        &

        chunk["A2"].isin(
            valid_alleles
        )

        &

        (
            chunk["A1"]
            !=
            chunk["A2"]
        )

        &

        valid_effect_allele.loc[
            chunk.index
        ]
    )


    stats["removed_invalid_alleles"] += (
        (~allele_mask).sum()
    )


    chunk = chunk[allele_mask]


    # NUMERIC CONVERSION

    numeric_columns = [

        "CHR",
        "BP",
        "FREQ",
        "BETA",
        "SE",
        "P",
        "N",
    ]


    for col in numeric_columns:

        chunk[col] = pd.to_numeric(

            chunk[col],

            errors="coerce"
        )

    # CHROMOSOME

    chr_mask = (

        np.isfinite(
            chunk["CHR"]
        )

        &

        (
            chunk["CHR"] >= 1
        )

        &

        (
            chunk["CHR"] <= 22
        )

        &

        (
            chunk["CHR"]
            ==
            np.floor(
                chunk["CHR"]
            )
        )
    )


    stats["removed_invalid_chr"] += (
        (~chr_mask).sum()
    )


    chunk = chunk[chr_mask]


    # BASE-PAIR POSITION

    bp_mask = (

        np.isfinite(
            chunk["BP"]
        )

        &

        (
            chunk["BP"] > 0
        )

        &

        (
            chunk["BP"]
            ==
            np.floor(
                chunk["BP"]
            )
        )
    )


    stats["removed_invalid_bp"] += (
        (~bp_mask).sum()
    )


    chunk = chunk[bp_mask]


    # EFFECT ALLELE FREQUENCY

    freq_mask = (

        np.isfinite(
            chunk["FREQ"]
        )

        &

        (
            chunk["FREQ"] >= 0
        )

        &

        (
            chunk["FREQ"] <= 1
        )
    )


    stats["removed_invalid_freq"] += (
        (~freq_mask).sum()
    )


    chunk = chunk[freq_mask]


    # ODDS RATIO
    #
    # OR must be positive because:

    #     BETA = ln(OR)
    #
    # is undefined for OR <= 0.

    or_mask = (

        np.isfinite(
            chunk["OR"]
        )

        &

        (
            chunk["OR"] > 0
        )
    )


    stats["removed_invalid_or"] += (
        (~or_mask).sum()
    )


    chunk = chunk[or_mask]

    # BETA

    beta_mask = np.isfinite(
        chunk["BETA"]
    )


    stats["removed_invalid_beta"] += (
        (~beta_mask).sum()
    )


    chunk = chunk[beta_mask]

    # STANDARD ERROR

    # SE must be positive and finite.

    se_mask = (

        np.isfinite(
            chunk["SE"]
        )

        &

        (
            chunk["SE"] > 0
        )
    )


    stats["removed_invalid_se"] += (
        (~se_mask).sum()
    )


    chunk = chunk[se_mask]


    # P-VALUE

    p_mask = (

        np.isfinite(
            chunk["P"]
        )

        &

        (
            chunk["P"] > 0
        )

        &

        (
            chunk["P"] <= 1
        )
    )


    stats["removed_invalid_p"] += (
        (~p_mask).sum()
    )


    chunk = chunk[p_mask]


    # SAMPLE SIZE

    n_mask = (

        np.isfinite(
            chunk["N"]
        )

        &

        (
            chunk["N"] > 0
        )
    )


    stats["removed_invalid_n"] += (
        (~n_mask).sum()
    )


    chunk = chunk[n_mask]


    # GLOBAL DUPLICATE SNP REMOVAL

    duplicate_mask = (
        chunk["SNP"].isin(
            seen_snps
        )
    )


    stats["removed_duplicate_snp"] += (
        duplicate_mask.sum()
    )


    chunk = chunk[
        ~duplicate_mask
    ]


    seen_snps.update(
        chunk["SNP"].tolist()
    )


    # 10F. RETAIN STANDARDIZED QC COLUMNS

    chunk = chunk[
        STANDARDIZED_COLUMNS
    ]

    # 10G. WRITE QC-PASSED CHUNK

    if len(chunk) > 0:

        mode = (

            "wt"

            if first_write

            else "at"
        )


        with gzip.open(

            LOCAL_QC_OUTPUT,

            mode,

            encoding="utf-8",

            newline=""
        ) as gz:

            chunk.to_csv(

                gz,

                sep="\t",

                index=False,

                header=first_write
            )


        first_write = False


        stats["qc_passed_rows"] += (
            len(chunk)
        )


    # Progress

    stats["chunks_completed"] = (
        chunk_number
    )


    elapsed = (

        time.time()

        -

        start_time

    ) / 60


    print(

        f"Chunk {chunk_number:03d} | "

        f"Input: "
        f"{stats['input_rows']:,} | "

        f"QC passed: "
        f"{stats['qc_passed_rows']:,} | "

        f"Elapsed: "
        f"{elapsed:.1f} min"
    )


# 11. FINAL QC STATISTICS

stats["runtime_minutes"] = (

    time.time()

    -

    start_time

) / 60


# 12. CONVERT NUMPY VALUES TO STANDARD PYTHON TYPES

stats_clean = {}


for key, value in stats.items():

    if isinstance(
        value,
        np.integer
    ):

        stats_clean[key] = int(
            value
        )


    elif isinstance(
        value,
        np.floating
    ):

        stats_clean[key] = float(
            value
        )


    else:

        stats_clean[key] = value


# 13. VERIFY LOCAL QC FILE

print(
    "\n" + "=" * 70
)

print(
    "VERIFYING LOCAL QC FILE"
)

print(
    "=" * 70
)


if not LOCAL_QC_OUTPUT.exists():

    raise RuntimeError(
        "Local QC output was not created."
    )


if LOCAL_QC_OUTPUT.stat().st_size == 0:

    raise RuntimeError(
        "Local QC output is empty."
    )


decompressed_bytes = 0


with gzip.open(
    LOCAL_QC_OUTPUT,
    "rb"
) as f:

    while True:

        data = f.read(
            4 * 1024 * 1024
        )


        if not data:

            break


        decompressed_bytes += (
            len(data)
        )


print(
    "Local gzip integrity check passed."
)


print(

    f"Compressed size: "

    f"{LOCAL_QC_OUTPUT.stat().st_size / (1024**3):.2f} GB"
)


print(

    f"Decompressed bytes verified: "

    f"{decompressed_bytes:,}"
)


# 14. COPY VERIFIED QC FILE TO GOOGLE DRIVE

print(
    "\nCopying verified QC file "
    "to Google Drive..."
)


# Confirm the destination directory exists.

GWAS_QC_DIR.mkdir(
    parents=True,
    exist_ok=True
)


shutil.copy2(
    LOCAL_QC_OUTPUT,
    QC_OUTPUT
)


print(
    "Copy completed."
)


print(
    f"QC file copied to:\n{QC_OUTPUT}"
)

# 15. VERIFY GOOGLE DRIVE COPY

print(
    "\nVerifying Google Drive copy..."
)


if not QC_OUTPUT.exists():

    raise RuntimeError(
        "QC file was not found on Google Drive "
        "after copying."
    )


drive_decompressed_bytes = 0


with gzip.open(
    QC_OUTPUT,
    "rb"
) as f:

    while True:

        data = f.read(
            4 * 1024 * 1024
        )


        if not data:

            break


        drive_decompressed_bytes += (
            len(data)
        )


if (
    decompressed_bytes
    != drive_decompressed_bytes
):

    raise RuntimeError(

        "Drive copy verification failed: "
        "decompressed byte count differs."
    )


print(
    "Google Drive gzip integrity check passed."
)


print(

    f"Verified decompressed bytes: "

    f"{drive_decompressed_bytes:,}"
)

# 16. SAVE QC REPORT

stats_clean["output_file"] = str(
    QC_OUTPUT
)


stats_clean[
    "verified_decompressed_bytes"
] = int(
    drive_decompressed_bytes
)


stats_clean[
    "standardized_columns"
] = STANDARDIZED_COLUMNS


stats_clean[
    "se_provenance"
] = (

    "SE was derived from the reported 95% confidence "
    "interval using "
    "SE = [ln(CI_upper) - ln(CI_lower)] / (2 × 1.96)."
)


stats_clean[
    "beta_provenance"
] = (

    "BETA was derived from the reported odds ratio "
    "using BETA = ln(OR)."
)


with open(
    QC_REPORT,
    "w"
) as f:

    json.dump(

        stats_clean,

        f,

        indent=2
    )


# 17. FINAL SUMMARY

print(
    "\nGWAS STANDARDIZATION + QC COMPLETE"
)


print(
    json.dumps(
        stats_clean,
        indent=2
    )
)


print(

    f"\nQC output size: "

    f"{QC_OUTPUT.stat().st_size / (1024**3):.2f} GB"
)


print(
    "\nQC output:"
)


print(
    QC_OUTPUT
)


print(
    "\nQC report:"
)


print(
    QC_REPORT
)

T2D GWAS SUMMARY STATISTICS — EUROPEAN ANCESTRY
STANDARDIZATION + SE DERIVATION + QUALITY CONTROL

Input file:
/content/drive/MyDrive/PRS_T2D_E-A/GWAS/MVP_T2D_Phe_250_2_EUR_raw.txt.gz

Input compressed size: 0.54 GB

Chunk size: 100,000
Local QC output:
/content/MVP_T2D_Phe_250_2_EUR_QC.tsv.gz
STANDARDIZATION PREVIEW
        SNP CHR     BP A1 A2   FREQ     OR      BETA  CI_LOWER  CI_UPPER       SE  SE_DERIVED      P      N
rs558358031   1  60879  A  G 0.9994 0.9731 -0.027268    0.6348     1.492 0.218001        True 0.9003 432648
rs562476259   1  85941  C  T 0.9987 1.1670  0.154436    0.8830     1.543 0.142387        True 0.2775 432648
rs567575968   1 133166  G  A 0.9998 0.9692 -0.031284    0.5685     1.652 0.272128        True 0.9085 432648
rs573978839   1 268598  G  T 0.9995 1.0190  0.018822    0.6642     1.564 0.218474        True 0.9304 432648
rs531691745   1 284844  A  C 0.9980 0.9005 -0.104805    0.7179     1.130 0.115725        True 0.3647 432648

SE DERIVATION CHECK
SE source: r

## 11. Harmonization of the QC-passed discovery GWAS

The QC-passed GWAS is converted into a standardized internal representation.

The harmonized dataset uses:

- `SNP` — variant identifier;
- `CHR` — chromosome;
- `BP` — base-pair position;
- `A1` — effect allele;
- `A2` — other allele;
- `EAF` — effect-allele frequency;
- `BETA` — effect estimate;
- `SE` — standard error;
- `P` — association P-value; and
- `N` — sample size where available.

At this stage, target-genotype-specific allele harmonization is not performed because an independent target genotype dataset is not yet available.

In [14]:
# EUR GWAS HARMONIZATION — SAFE SINGLE-STREAM GZIP

from pathlib import Path
import gzip
import shutil
import time
import pandas as pd


# 1. PATHS

BASE_DIR = Path(
    "/content/drive/MyDrive/PRS_T2D_E-A"
)

QC_DIR = (
    BASE_DIR / "results" / "gwas_qc"
)

HARMONIZED_DIR = (
    BASE_DIR / "results" / "harmonized"
)

QC_OUTPUT = (
    QC_DIR /
    "MVP_T2D_Phe_250_2_EUR_QC.tsv.gz"
)

HARMONIZED_OUTPUT = (
    HARMONIZED_DIR /
    "MVP_T2D_Phe_250_2_EUR_harmonized.tsv.gz"
)

LOCAL_HARMONIZED_OUTPUT = Path(
    "/content/MVP_T2D_Phe_250_2_EUR_harmonized.tsv.gz"
)


# Create output directory if it does not exist

HARMONIZED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# 2. CHECK INPUT

if not QC_OUTPUT.exists():
    raise FileNotFoundError(
        f"QC file not found:\n{QC_OUTPUT}"
    )

print("=" * 70)
print("EUR GWAS HARMONIZATION")
print("=" * 70)
print(f"Input QC file:")
print(QC_OUTPUT)
print()
print(f"Local output:")
print(LOCAL_HARMONIZED_OUTPUT)
print()
print(f"Drive output:")
print(HARMONIZED_OUTPUT)
print()


# 3. COLUMNS

input_columns = [
    "CHR",
    "BP",
    "SNP",
    "A1",
    "A2",
    "FREQ",
    "BETA",
    "SE",
    "P",
    "N"
]

output_columns = [
    "SNP",
    "CHR",
    "BP",
    "A1",
    "A2",
    "EAF",
    "BETA",
    "SE",
    "P",
    "N"
]


# 4. REMOVE PREVIOUS LOCAL OUTPUT

if LOCAL_HARMONIZED_OUTPUT.exists():
    LOCAL_HARMONIZED_OUTPUT.unlink()

first_write = True
total_rows = 0
chunk_number = 0
start_time = time.time()


# 5. PROCESS QC FILE

print("Starting harmonization...")
print()

for chunk_number, chunk in enumerate(
    pd.read_csv(
        QC_OUTPUT,
        sep="\t",
        compression="gzip",
        usecols=input_columns,
        chunksize=100_000,
        low_memory=False
    ),
    start=1
):

    # Rename allele frequency

    chunk = chunk.rename(
        columns={
            "FREQ": "EAF"
        }
    )


    # Standardize string columns

    chunk["SNP"] = (
        chunk["SNP"]
        .astype(str)
        .str.strip()
    )

    chunk["A1"] = (
        chunk["A1"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    chunk["A2"] = (
        chunk["A2"]
        .astype(str)
        .str.upper()
        .str.strip()
    )


    # Standardize numeric columns

    for col in [
        "CHR",
        "BP",
        "EAF",
        "BETA",
        "SE",
        "P",
        "N"
    ]:

        chunk[col] = pd.to_numeric(
            chunk[col],
            errors="coerce"
        )

    # Final column order
    # --------------------------------------------------------

    chunk = chunk[
        output_columns
    ]


    # --------------------------------------------------------
    # Write to ONE local gzip stream
    # --------------------------------------------------------

    mode = (
        "wt"
        if first_write
        else "at"
    )

    with gzip.open(
        LOCAL_HARMONIZED_OUTPUT,
        mode,
        encoding="utf-8",
        newline=""
    ) as gz:

        chunk.to_csv(
            gz,
            sep="\t",
            index=False,
            header=first_write
        )


    first_write = False
    total_rows += len(chunk)

    elapsed = (
        time.time() - start_time
    ) / 60

    print(
        f"Chunk {chunk_number:03d} | "
        f"Harmonized: {total_rows:,} variants | "
        f"Elapsed: {elapsed:.1f} min"
    )


# ============================================================
# 6. LOCAL OUTPUT INTEGRITY CHECK
# ============================================================

print()
print("=" * 70)
print("LOCAL OUTPUT CHECK")
print("=" * 70)

if not LOCAL_HARMONIZED_OUTPUT.exists():
    raise RuntimeError(
        "Local harmonized output was not created."
    )

local_size = (
    LOCAL_HARMONIZED_OUTPUT.stat().st_size
)

print(
    f"Local file size: "
    f"{local_size / (1024**3):.2f} GB"
)


# ------------------------------------------------------------
# Check gzip integrity
# ------------------------------------------------------------

decompressed_bytes = 0

with gzip.open(
    LOCAL_HARMONIZED_OUTPUT,
    "rb"
) as gz:

    while True:

        block = gz.read(
            1024 * 1024
        )

        if not block:
            break

        decompressed_bytes += len(block)


print(
    f"Decompressed bytes verified: "
    f"{decompressed_bytes:,}"
)


# 7. COPY TO GOOGLE DRIVE

print()
print("COPYING TO GOOGLE DRIVE")


shutil.copy2(
    LOCAL_HARMONIZED_OUTPUT,
    HARMONIZED_OUTPUT
)

print(
    f"Copied to:\n"
    f"{HARMONIZED_OUTPUT}"
)


# 8. VERIFY DRIVE COPY


local_size = (
    LOCAL_HARMONIZED_OUTPUT.stat().st_size
)

drive_size = (
    HARMONIZED_OUTPUT.stat().st_size
)

print()
print("Local size :", local_size)
print("Drive size :", drive_size)

if local_size != drive_size:
    raise RuntimeError(
        "ERROR: Local and Drive file sizes do not match."
    )

print(
    "Drive copy verified successfully."
)


# 9. FINAL SUMMARY

elapsed_total = (
    time.time() - start_time
) / 60

print()

print("HARMONIZATION COMPLETE")


print(
    f"Total variants: "
    f"{total_rows:,}"
)

print(
    f"Chunks processed: "
    f"{chunk_number:,}"
)

print(
    f"Final output columns:\n"
    f"{output_columns}"
)

print()
print(
    f"Local file:\n"
    f"{LOCAL_HARMONIZED_OUTPUT}"
)

print()
print(
    f"Drive file:\n"
    f"{HARMONIZED_OUTPUT}"
)

print()
print(
    f"Total elapsed time: "
    f"{elapsed_total:.1f} minutes"
)

print()
print("Status: SUCCESS")

EUR GWAS HARMONIZATION
Input QC file:
/content/drive/MyDrive/PRS_T2D_E-A/results/gwas_qc/MVP_T2D_Phe_250_2_EUR_QC.tsv.gz

Local output:
/content/MVP_T2D_Phe_250_2_EUR_harmonized.tsv.gz

Drive output:
/content/drive/MyDrive/PRS_T2D_E-A/results/harmonized/MVP_T2D_Phe_250_2_EUR_harmonized.tsv.gz

Starting harmonization...

Chunk 001 | Harmonized: 100,000 variants | Elapsed: 0.1 min
Chunk 002 | Harmonized: 200,000 variants | Elapsed: 0.1 min
Chunk 003 | Harmonized: 300,000 variants | Elapsed: 0.2 min
Chunk 004 | Harmonized: 400,000 variants | Elapsed: 0.2 min
Chunk 005 | Harmonized: 500,000 variants | Elapsed: 0.3 min
Chunk 006 | Harmonized: 600,000 variants | Elapsed: 0.3 min
Chunk 007 | Harmonized: 700,000 variants | Elapsed: 0.4 min
Chunk 008 | Harmonized: 800,000 variants | Elapsed: 0.4 min
Chunk 009 | Harmonized: 900,000 variants | Elapsed: 0.5 min
Chunk 010 | Harmonized: 1,000,000 variants | Elapsed: 0.5 min
Chunk 011 | Harmonized: 1,100,000 variants | Elapsed: 0.6 min
Chunk 012 | Ha

## 12. Generate candidate SNP sets at predefined GWAS P-value thresholds

Variants are selected using six predefined association P-value thresholds:

- P ≤ 5 × 10⁻⁸
- P ≤ 1 × 10⁻⁵
- P ≤ 1 × 10⁻⁴
- P ≤ 1 × 10⁻³
- P ≤ 1 × 10⁻²
- P ≤ 5 × 10⁻²

These candidate sets are subsequently subjected to LD clumping using the corresponding threshold-specific GWAS significance level.

In [15]:
from pathlib import Path
import time
import pandas as pd

# 1. PATHS

BASE_DIR = Path(
    "/content/drive/MyDrive/PRS_T2D_E-A"
)

HARMONIZED_OUTPUT = (
    BASE_DIR
    / "results"
    / "harmonized"
    / "MVP_T2D_Phe_250_2_EUR_harmonized.tsv.gz"
)

CANDIDATE_DIR = (
    BASE_DIR
    / "results"
    / "candidate_snps"
)

CANDIDATE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Check that the harmonized GWAS exists

if not HARMONIZED_OUTPUT.exists():
    raise FileNotFoundError(
        f"Harmonized GWAS file not found:\n{HARMONIZED_OUTPUT}"
    )

# 2. P-VALUE THRESHOLDS

P_THRESHOLDS = [
    5e-8,
    1e-5,
    1e-4,
    1e-3,
    1e-2,
    5e-2,
]

# 3. CLEAN PREVIOUS CANDIDATE FILES

for threshold in P_THRESHOLDS:

    output_path = (
        CANDIDATE_DIR
        / f"MVP_T2D_Phe_250_2_EUR_P{threshold:g}.tsv"
    )

    if output_path.exists():
        output_path.unlink()

# 4. INITIALIZE COUNTS

candidate_counts = {
    threshold: 0
    for threshold in P_THRESHOLDS
}

start_time = time.time()

print("CREATING EUR CANDIDATE SNP SETS")
print(f"Input: {HARMONIZED_OUTPUT}")
print(f"Output directory: {CANDIDATE_DIR}")
print()

# 5. READ HARMONIZED GWAS IN CHUNKS

for chunk_number, chunk in enumerate(
    pd.read_csv(
        HARMONIZED_OUTPUT,
        sep="\t",
        compression="gzip",
        chunksize=100_000,
        low_memory=False,
    ),
    start=1
):

    chunk["P"] = pd.to_numeric(
        chunk["P"],
        errors="coerce"
    )

    # Select SNPs at each P-value threshold

    for threshold in P_THRESHOLDS:

        selected = chunk[
            chunk["P"] <= threshold
        ]

        if selected.empty:
            continue

        output_path = (
            CANDIDATE_DIR
            / f"MVP_T2D_Phe_250_2_EUR_P{threshold:g}.tsv"
        )

        selected.to_csv(
            output_path,
            sep="\t",
            index=False,
            mode="a",
            header=not output_path.exists(),
        )

        candidate_counts[threshold] += len(selected)

    if chunk_number % 10 == 0:

        elapsed = (
            time.time() - start_time
        ) / 60

        print(
            f"Chunks processed: {chunk_number:,} | "
            f"Elapsed: {elapsed:.1f} min"
        )

# 6. FINAL SUMMARY

print()
print("EUR CANDIDATE SNP SETS CREATED")
print()

for threshold in P_THRESHOLDS:

    output_path = (
        CANDIDATE_DIR
        / f"MVP_T2D_Phe_250_2_EUR_P{threshold:g}.tsv"
    )

    count = candidate_counts[threshold]

    if count > 0 and output_path.exists():

        print(
            f"P <= {threshold:g}: "
            f"{count:,} SNPs | "
            f"{output_path.name}"
        )

    else:

        print(
            f"P <= {threshold:g}: 0 SNPs"
        )

print()
print(
    f"Runtime: "
    f"{(time.time() - start_time) / 60:.2f} minutes"
)

print(
    f"Output directory: {CANDIDATE_DIR}"
)

CREATING EUR CANDIDATE SNP SETS
Input: /content/drive/MyDrive/PRS_T2D_E-A/results/harmonized/MVP_T2D_Phe_250_2_EUR_harmonized.tsv.gz
Output directory: /content/drive/MyDrive/PRS_T2D_E-A/results/candidate_snps

Chunks processed: 10 | Elapsed: 0.1 min
Chunks processed: 20 | Elapsed: 0.2 min
Chunks processed: 30 | Elapsed: 0.3 min
Chunks processed: 40 | Elapsed: 0.5 min
Chunks processed: 50 | Elapsed: 0.6 min
Chunks processed: 60 | Elapsed: 0.7 min
Chunks processed: 70 | Elapsed: 0.8 min
Chunks processed: 80 | Elapsed: 0.9 min
Chunks processed: 90 | Elapsed: 1.0 min
Chunks processed: 100 | Elapsed: 1.2 min
Chunks processed: 110 | Elapsed: 1.2 min
Chunks processed: 120 | Elapsed: 1.4 min
Chunks processed: 130 | Elapsed: 1.5 min
Chunks processed: 140 | Elapsed: 1.6 min
Chunks processed: 150 | Elapsed: 1.7 min
Chunks processed: 160 | Elapsed: 1.8 min
Chunks processed: 170 | Elapsed: 1.9 min
Chunks processed: 180 | Elapsed: 2.0 min

EUR CANDIDATE SNP SETS CREATED

P <= 5e-08: 67,024 SNPs | MV

In [16]:
# Combine all EUR candidate SNP TSVs

from pathlib import Path
import pandas as pd

# 1. PATHS

CANDIDATE_DIR = Path(
    "/content/drive/MyDrive/PRS_T2D_E-A/results/candidate_snps"
)

COMBINED_CANDIDATE_FILE = (
    CANDIDATE_DIR
    / "MVP_T2D_Phe_250_2_EUR_all_candidate_SNPs.tsv.gz"
)

P_THRESHOLDS = [
    5e-8,
    1e-5,
    1e-4,
    1e-3,
    1e-2,
    5e-2,
]

# 2. READ AND COMBINE ALL THRESHOLDS

candidate_frames = []

for threshold in P_THRESHOLDS:

    input_path = (
        CANDIDATE_DIR
        / f"MVP_T2D_Phe_250_2_EUR_P{threshold:g}.tsv"
    )

    if not input_path.exists():
        raise FileNotFoundError(
            f"Candidate file not found: {input_path}"
        )

    df = pd.read_csv(
        input_path,
        sep="\t",
        usecols=["SNP", "P"]
    )

    candidate_frames.append(df)

# 3. COMBINE

combined = pd.concat(
    candidate_frames,
    ignore_index=True
)

# 4. CLEAN AND DEDUPLICATE

combined["SNP"] = (
    combined["SNP"]
    .astype(str)
    .str.strip()
)

combined["P"] = pd.to_numeric(
    combined["P"],
    errors="coerce"
)

combined = (
    combined
    .dropna(subset=["SNP", "P"])
    .sort_values("P")
    .drop_duplicates(subset=["SNP"])
)

# 5. SAVE

combined.to_csv(
    COMBINED_CANDIDATE_FILE,
    sep="\t",
    index=False,
    compression="gzip"
)

print("Combined EUR candidate SNP file created:")
print(COMBINED_CANDIDATE_FILE)
print(f"Unique candidate SNPs: {len(combined):,}")

Combined EUR candidate SNP file created:
/content/drive/MyDrive/PRS_T2D_E-A/results/candidate_snps/MVP_T2D_Phe_250_2_EUR_all_candidate_SNPs.tsv.gz
Unique candidate SNPs: 3,066,477


## 13. Prepare threshold-specific LD-clumping inputs

For each GWAS P-value threshold, a dedicated clumping input file is created.

Each input contains the variant information required by PLINK2 for LD clumping, including:

- SNP identifier;
- chromosome;
- genomic position;
- effect allele;
- other allele; and
- association P-value.

Separate input files allow each threshold to be processed independently and make the clumping stage reproducible.

In [17]:
# Prepare PLINK LD-clumping inputs with SNP, A1 and P

from pathlib import Path
import pandas as pd

# 1. PATHS

BASE_DIR = Path(
    "/content/drive/MyDrive/PRS_T2D_E-A"
)

CANDIDATE_DIR = (
    BASE_DIR
    / "results"
    / "candidate_snps"
)

CLUMP_DIR = (
    BASE_DIR
    / "results"
    / "ld_clumping"
)

CLUMP_DIR.mkdir(
    parents=True,
    exist_ok=True
)

P_THRESHOLDS = [
    5e-8,
    1e-5,
    1e-4,
    1e-3,
    1e-2,
    5e-2,
]

# 2. CREATE CLUMPING INPUTS

print("PREPARING PLINK LD-CLUMPING INPUTS")
print()

for threshold in P_THRESHOLDS:

    candidate_file = (
        CANDIDATE_DIR
        / f"MVP_T2D_Phe_250_2_EUR_P{threshold:g}.tsv"
    )

    clump_file = (
        CLUMP_DIR
        / f"MVP_T2D_Phe_250_2_EUR_P{threshold:g}_clump_input.txt"
    )

    print(f"Processing P <= {threshold:g}")
    print(f"Source: {candidate_file}")

    if not candidate_file.exists():
        raise FileNotFoundError(
            f"Candidate file missing:\n{candidate_file}"
        )

    # Read the three columns required for PLINK clumping

    df = pd.read_csv(
        candidate_file,
        sep="\t",
        usecols=["SNP", "A1", "P"],
        dtype={
            "SNP": str,
            "A1": str
        }
    )

    print(
        f"Rows loaded: {len(df):,}"
    )

    # Remove rows missing required fields

    before = len(df)

    df = df.dropna(
        subset=["SNP", "A1", "P"]
    )

    removed = before - len(df)

    if removed > 0:
        print(
            f"Rows removed for missing "
            f"SNP/A1/P: {removed:,}"
        )

    # Standardize SNP and allele strings

    df["SNP"] = (
        df["SNP"]
        .astype(str)
        .str.strip()
    )

    df["A1"] = (
        df["A1"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    # Make sure P is numeric

    df["P"] = pd.to_numeric(
        df["P"],
        errors="coerce"
    )

    before = len(df)

    df = df.dropna(
        subset=["P"]
    )

    removed_p = before - len(df)

    if removed_p > 0:
        print(
            f"Rows removed for invalid P: "
            f"{removed_p:,}"
        )

    # Remove duplicate SNP IDs

    before = len(df)

    df = (
        df
        .sort_values("P")
        .drop_duplicates(
            subset=["SNP"],
            keep="first"
        )
    )

    removed_dup = before - len(df)

    if removed_dup > 0:
        print(
            f"Duplicate SNPs removed: "
            f"{removed_dup:,}"
        )

    # Keep exactly the columns needed by PLINK

    df = df[
        ["SNP", "A1", "P"]
    ]

    # Write clumping input

    df.to_csv(
        clump_file,
        sep="\t",
        index=False
    )

    print(
        f"Written: {clump_file}"
    )

    print(
        f"Final rows: {len(df):,}"
    )

    print(
        f"Columns: {list(df.columns)}"
    )

    print()


# 3. VERIFY ALL FILES

print("VERIFYING CLUMPING INPUT FILES")
print()

for threshold in P_THRESHOLDS:

    clump_file = (
        CLUMP_DIR
        / f"MVP_T2D_Phe_250_2_EUR_P{threshold:g}_clump_input.txt"
    )

    if not clump_file.exists():
        print(
            f"P <= {threshold:g}: MISSING"
        )
        continue

    check = pd.read_csv(
        clump_file,
        sep="\t",
        nrows=5
    )

    print(
        f"P <= {threshold:g}: "
        f"{list(check.columns)} | "
        f"rows readable"
    )

print()
print("PLINK CLUMPING INPUTS READY")
print(
    f"Directory: {CLUMP_DIR}"
)

PREPARING PLINK LD-CLUMPING INPUTS

Processing P <= 5e-08
Source: /content/drive/MyDrive/PRS_T2D_E-A/results/candidate_snps/MVP_T2D_Phe_250_2_EUR_P5e-08.tsv
Rows loaded: 67,024
Duplicate SNPs removed: 28
Written: /content/drive/MyDrive/PRS_T2D_E-A/results/ld_clumping/MVP_T2D_Phe_250_2_EUR_P5e-08_clump_input.txt
Final rows: 66,996
Columns: ['SNP', 'A1', 'P']

Processing P <= 1e-05
Source: /content/drive/MyDrive/PRS_T2D_E-A/results/candidate_snps/MVP_T2D_Phe_250_2_EUR_P1e-05.tsv
Rows loaded: 161,307
Duplicate SNPs removed: 71
Written: /content/drive/MyDrive/PRS_T2D_E-A/results/ld_clumping/MVP_T2D_Phe_250_2_EUR_P1e-05_clump_input.txt
Final rows: 161,236
Columns: ['SNP', 'A1', 'P']

Processing P <= 0.0001
Source: /content/drive/MyDrive/PRS_T2D_E-A/results/candidate_snps/MVP_T2D_Phe_250_2_EUR_P0.0001.tsv
Rows loaded: 275,109
Duplicate SNPs removed: 134
Written: /content/drive/MyDrive/PRS_T2D_E-A/results/ld_clumping/MVP_T2D_Phe_250_2_EUR_P0.0001_clump_input.txt
Final rows: 274,975
Columns: [

## 14. Obtain and inspect 1000 Genomes Phase 3 sample metadata

The 1000 Genomes Phase 3 sample metadata are used to identify individuals belonging to African population groups.

The African reference panel includes:

- ACB
- ASW
- ESN
- GWD
- LWK
- MSL
- YRI

The resulting African sample list is saved and reused throughout the pipeline.

The 1000 Genomes Phase 3 genotype files (all_phase3.pgen.zst and all_phase3.pvar.zst) were downloaded manually from the PLINK 2.0 Resources page because their large file sizes caused automated downloads through Google Colab to fail. The sample metadata file (phase3_corrected.psam) was downloaded programmatically and saved as all_phase3.psam.
https://www.cog-genomics.org/plink/2.0/resources



In [18]:
import pandas as pd
from pathlib import Path

# 14. OBTAIN AND INSPECT 1000 GENOMES PHASE 3 SAMPLE METADATA

REFERENCE_DIR = Path(
    "/content/drive/MyDrive/PRS_T2D_E-A/reference"
)

PSAM_FILE = (
    REFERENCE_DIR / "all_phase3.psam"
)

AFR_SAMPLE_FILE = (
    REFERENCE_DIR / "AFR_samples.tsv"
)

# Check that the existing PSAM file is available

if not PSAM_FILE.exists():
    raise FileNotFoundError(
        f"1000 Genomes sample metadata not found: {PSAM_FILE}"
    )

print("Loading 1000 Genomes Phase 3 sample metadata...")
print(f"File: {PSAM_FILE}")

print(
    f"File size: "
    f"{PSAM_FILE.stat().st_size / 1024:.1f} KB"
)

psam = pd.read_csv(
    PSAM_FILE,
    sep="\t"
)

print("Shape:", psam.shape)
print("Columns:", psam.columns.tolist())

display(psam.head())

# Identify 1000 Genomes African-superpopulation samples

afr_psam = psam[
    psam["SuperPop"] == "AFR"
].copy()

print(
    "Total 1000 Genomes samples:",
    len(psam)
)

print(
    "African (AFR) samples:",
    len(afr_psam)
)

print("African populations:")

print(
    afr_psam["Population"]
    .value_counts()
    .sort_index()
)

print("First 10 AFR samples:")

display(
    afr_psam.head(10)
)

# Save African sample metadata

afr_psam.to_csv(
    AFR_SAMPLE_FILE,
    sep="\t",
    index=False
)

print(
    f"African sample metadata saved to: "
    f"{AFR_SAMPLE_FILE}"
)

Loading 1000 Genomes Phase 3 sample metadata...
File: /content/drive/MyDrive/PRS_T2D_E-A/reference/all_phase3.psam
File size: 53.9 KB
Shape: (2504, 6)
Columns: ['#IID', 'PAT', 'MAT', 'SEX', 'SuperPop', 'Population']


,#IID,PAT,MAT,SEX,SuperPop,Population
0,HG00096,0,0,1.0,EUR,GBR
1,HG00097,0,0,2.0,EUR,GBR
2,HG00099,0,0,2.0,EUR,GBR
3,HG00100,0,0,2.0,EUR,GBR
4,HG00101,0,0,1.0,EUR,GBR


Total 1000 Genomes samples: 2504
African (AFR) samples: 661
African populations:
Population
ACB     96
ASW     61
ESN     99
GWD    113
LWK     99
MSL     85
YRI    108
Name: count, dtype: int64
First 10 AFR samples:


,#IID,PAT,MAT,SEX,SuperPop,Population
673,HG01879,0,0,1.0,AFR,ACB
674,HG01880,0,0,2.0,AFR,ACB
675,HG01882,0,0,1.0,AFR,ACB
676,HG01883,0,0,2.0,AFR,ACB
677,HG01885,0,0,1.0,AFR,ACB
678,HG01886,0,0,2.0,AFR,ACB
679,HG01889,0,0,2.0,AFR,ACB
680,HG01890,0,0,1.0,AFR,ACB
683,HG01894,0,0,2.0,AFR,ACB
684,HG01896,0,0,2.0,AFR,ACB


African sample metadata saved to: /content/drive/MyDrive/PRS_T2D_E-A/reference/AFR_samples.tsv


## Define the African 1000 Genomes reference sample
Select individuals belonging to the AFR superpopulation in 1000 Genomes Phase 3 and record the population composition.
The resulting keep file defines the 661-sample African-ancestry LD reference called the AFR keep file.

In [19]:
AFR_KEEP = REFERENCE_DIR / "1000G_AFR.keep"

afr_psam[["#IID"]].assign(FID=0)[["FID", "#IID"]].to_csv(
    AFR_KEEP,
    sep="\t",
    index=False,
    header=False
)

print("AFR keep file created:")
print(AFR_KEEP)

print("\nNumber of samples in keep file:", len(afr_psam))

AFR keep file created:
/content/drive/MyDrive/PRS_T2D_E-A/reference/1000G_AFR.keep

Number of samples in keep file: 661


##15. Prepare the 1000 Genomes Phase 3 African LD reference
The 1000 Genomes Phase 3 reference genotype dataset is used as the LD reference panel. The required persistent files are stored in the project reference directory:
- all_phase3.pgen.zst
- all_phase3.pvar.zst
- all_phase3.psam

The African sample list is stored as:
- 1000G_AFR.keep

In [20]:
from pathlib import Path

REFERENCE_DIR = Path(
    "/content/drive/MyDrive/PRS_T2D_E-A/reference"
)

for filename in [
    "all_phase3.pgen.zst",
    "all_phase3.pvar.zst",
    "all_phase3.psam",
    "1000G_AFR.keep",
]:

    path = REFERENCE_DIR / filename

    print(f"\n{filename}")
    print("Exists:", path.exists())

    if path.exists():
        print(
            "Size:",
            round(
                path.stat().st_size / (1024**3),
                2
            ),
            "GB"
        )


all_phase3.pgen.zst
Exists: True
Size: 2.25 GB

all_phase3.pvar.zst
Exists: True
Size: 0.6 GB

all_phase3.psam
Exists: True
Size: 0.0 GB

1000G_AFR.keep
Exists: True
Size: 0.0 GB


In [21]:
from pathlib import Path
import subprocess
import shutil

PLINK = "/content/plink2/plink2"

DRIVE_REF = Path(
    "/content/drive/MyDrive/PRS_T2D_E-A/reference"
)

LOCAL_REF = Path(
    "/content/1000G_AFR_reference"
)

LOCAL_REF.mkdir(
    parents=True,
    exist_ok=True
)

PGEN_ZST = DRIVE_REF / "all_phase3.pgen.zst"
PVAR_ZST = DRIVE_REF / "all_phase3.pvar.zst"
PSAM = DRIVE_REF / "all_phase3.psam"
AFR_KEEP = DRIVE_REF / "1000G_AFR.keep"

# Check persistent files

for path in [
    PGEN_ZST,
    PVAR_ZST,
    PSAM,
    AFR_KEEP,
]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing persistent reference file: {path}"
        )

print("Persistent 1000G files found.")

# Copy PVAR, PSAM and AFR keep file locally

for src in [
    PVAR_ZST,
    PSAM,
    AFR_KEEP,
]:

    dst = LOCAL_REF / src.name

    if not dst.exists():
        print(f"Copying {src.name}...")
        shutil.copy2(src, dst)

# Decompress PGEN

LOCAL_PGEN = LOCAL_REF / "all_phase3.pgen"

if LOCAL_PGEN.exists():
    print("Local PGEN already exists.")

else:

    print("Decompressing all_phase3.pgen.zst...")
    print("This may take a while.")

    cmd = [
        PLINK,
        "--zst-decompress",
        str(PGEN_ZST),
        str(LOCAL_PGEN),
    ]

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )

    if result.returncode != 0:

        print(result.stdout)
        print(result.stderr)

        raise RuntimeError(
            "PGEN decompression failed."
        )

    print("PGEN decompression completed.")

# Verify files

required_local = [
    LOCAL_REF / "all_phase3.pgen",
    LOCAL_REF / "all_phase3.pvar.zst",
    LOCAL_REF / "all_phase3.psam",
    LOCAL_REF / "1000G_AFR.keep",
]

for path in required_local:

    if not path.exists():
        raise FileNotFoundError(
            f"Expected local reference file missing: {path}"
        )

print("1000G AFR REFERENCE RESTORED")

for path in required_local:

    size_gb = path.stat().st_size / (1024**3)

    print(
        f"{path.name}: {size_gb:.2f} GB"
    )

print("\nReady for LD clumping.")

Persistent 1000G files found.
Copying all_phase3.pvar.zst...
Copying all_phase3.psam...
Copying 1000G_AFR.keep...
Decompressing all_phase3.pgen.zst...
This may take a while.
PGEN decompression completed.
1000G AFR REFERENCE RESTORED
all_phase3.pgen: 6.24 GB
all_phase3.pvar.zst: 0.60 GB
all_phase3.psam: 0.00 GB
1000G_AFR.keep: 0.00 GB

Ready for LD clumping.


##16. Construct the African-only LD reference
The complete 1000 Genomes Phase 3 reference is restricted to the predefined African-ancestry samples. This produces an African-only reference dataset that is used for LD estimation and clumping. The resulting files are stored persistently in the project reference directory.

In [25]:
from pathlib import Path
import subprocess

PLINK = "/content/plink2/plink2"

LOCAL_REF = Path("/content/1000G_AFR_reference")
AFR_KEEP = LOCAL_REF / "1000G_AFR.keep"
AFR_REF = LOCAL_REF / "AFR_only"

print("CREATING AFR-ONLY 1000 GENOMES REFERENCE")

cmd = [
    PLINK,
    "--pfile", str(LOCAL_REF / "all_phase3"), "vzs",
    "--keep", str(AFR_KEEP),
    "--make-pgen",
    "--out", str(AFR_REF),
]

print("Running PLINK2...")
print(" ".join(cmd))

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("AFR-only reference creation failed.")

print("\nAFR-only reference created.")

for suffix in [".pgen", ".pvar", ".psam"]:
    path = Path(str(AFR_REF) + suffix)

    if not path.exists():
        raise FileNotFoundError(f"Missing output: {path}")

    print(
        f"{path.name}: "
        f"{path.stat().st_size / (1024**2):.1f} MB"
    )

print("\nReady for LD clumping.")

CREATING AFR-ONLY 1000 GENOMES REFERENCE
Running PLINK2...
/content/plink2/plink2 --pfile /content/1000G_AFR_reference/all_phase3 vzs --keep /content/1000G_AFR_reference/1000G_AFR.keep --make-pgen --out /content/1000G_AFR_reference/AFR_only
PLINK v2.0.0-b.1-devLM 64-bit Intel (1 Oct 2026)    cog-genomics.org/plink/2.0/
(C) 2005-2026 S Purcell, C Chang, B Demaille      GNU General Public License v3
Logging to /content/1000G_AFR_reference/AFR_only.log.
Options in effect:
  --keep /content/1000G_AFR_reference/1000G_AFR.keep
  --make-pgen
  --out /content/1000G_AFR_reference/AFR_only
  --pfile /content/1000G_AFR_reference/all_phase3 vzs

Start time: Sun Oct  4 12:13:37 2026
12975 MiB RAM detected, ~6521 available; reserving 6457 MiB for main workspace.
Using up to 2 compute threads.
2504 samples (1270 females, 1233 males, 1 ambiguous; 2497 founders) loaded from
/content/1000G_AFR_reference/all_phase3.psam.
84805772 variants loaded from /content/1000G_AFR_reference/all_phase3.pvar.zst.
2 ca

##17. Verify the African-only LD reference
The generated African reference dataset is checked to ensure that the expected PLINK files exist and can be read successfully. The sample count and variant structure are recorded as a computational checkpoint.

In [26]:

from pathlib import Path
import shutil

LOCAL_AFR = Path("/content/1000G_AFR_reference/AFR_only")
DRIVE_AFR = Path(
    "/content/drive/MyDrive/PRS_T2D_E-A/reference/AFR_only"
)

DRIVE_AFR.parent.mkdir(parents=True, exist_ok=True)

for suffix in [".pgen", ".pvar", ".psam"]:
    src = Path(str(LOCAL_AFR) + suffix)
    dst = Path(str(DRIVE_AFR) + suffix)

    if not src.exists():
        raise FileNotFoundError(f"Missing local file: {src}")

    print(f"Copying {src.name} → Drive...")
    shutil.copy2(src, dst)

print("AFR-ONLY REFERENCE SAVED TO GOOGLE DRIVE")

for suffix in [".pgen", ".pvar", ".psam"]:
    path = Path(str(DRIVE_AFR) + suffix)
    print(
        f"{path.name}: "
        f"{path.stat().st_size / (1024**3):.2f} GB"
    )

print("\nPersistent copy complete.")

Copying AFR_only.pgen → Drive...
Copying AFR_only.pvar → Drive...
Copying AFR_only.psam → Drive...
AFR-ONLY REFERENCE SAVED TO GOOGLE DRIVE
AFR_only.pgen: 2.13 GB
AFR_only.pvar: 2.16 GB
AFR_only.psam: 0.00 GB

Persistent copy complete.


## 18. Create a candidate SNP lookup set
The union of all candidate SNPs from the European-ancestry GWAS across the six P-value thresholds is used to create a lookup set.

This lookup set is used to identify which European GWAS candidate variants are present in the African 1000 Genomes reference panel.

Using the union of candidate SNPs avoids repeatedly scanning the African reference for the same variants across the six P-value thresholds.

The candidate SNPs come from the European GWAS.

The reference PVAR comes from the African 1000 Genomes reference.

The output should identify which EUR-GWAS candidate SNPs are present in the AFR LD reference.

In [27]:
from pathlib import Path
import pandas as pd

# 1. PATHS

REFERENCE_DIR = Path(
    "/content/drive/MyDrive/PRS_T2D_E-A/reference"
)

CANDIDATE_DIR = Path(
    "/content/drive/MyDrive/PRS_T2D_E-A/results/candidate_snps"
)

# African-only reference prefix

AFR_ONLY = REFERENCE_DIR / "AFR_only"

# Combined candidate SNP file from the European GWAS

CANDIDATE_SNP_LIST = (
    CANDIDATE_DIR
    / "MVP_T2D_Phe_250_2_EUR_all_candidate_SNPs.tsv.gz"
)

# African-only reference PVAR

AFR_PVAR = Path(
    str(AFR_ONLY) + ".pvar"
)

# Output

OUTPUT_FILE = (
    CANDIDATE_DIR
    / "MVP_T2D_Phe_250_2_EUR_candidate_SNPs_in_1000G_AFR.txt"
)

print(
    "Candidate SNP list:",
    CANDIDATE_SNP_LIST
)

print(
    "AFR PVAR:",
    AFR_PVAR
)

# 2. CHECK REQUIRED FILES

if not CANDIDATE_SNP_LIST.exists():
    raise FileNotFoundError(
        f"Candidate SNP list not found: "
        f"{CANDIDATE_SNP_LIST}"
    )

if not AFR_PVAR.exists():
    raise FileNotFoundError(
        f"African reference PVAR not found: "
        f"{AFR_PVAR}"
    )

print("\nAll required input files exist.")

# 3. LOAD CANDIDATE SNPs

candidate_df = pd.read_csv(
    CANDIDATE_SNP_LIST,
    sep="\t",
    compression="gzip",
    usecols=["SNP"]
)

candidate_snps = set(
    candidate_df["SNP"]
    .dropna()
    .astype(str)
    .str.strip()
)

print(
    f"Candidate SNPs: "
    f"{len(candidate_snps):,}"
)

Candidate SNP list: /content/drive/MyDrive/PRS_T2D_E-A/results/candidate_snps/MVP_T2D_Phe_250_2_EUR_all_candidate_SNPs.tsv.gz
AFR PVAR: /content/drive/MyDrive/PRS_T2D_E-A/reference/AFR_only.pvar

All required input files exist.
Candidate SNPs: 3,066,477


## 19. Match EUR GWAS candidate SNPs to the African LD reference PVAR

The candidate variants from the European-ancestry GWAS are matched against the African 1000 Genomes reference PVAR.

The PVAR is scanned sequentially rather than loading the entire reference variant table into memory. This is necessary because the 1000 Genomes reference contains tens of millions of variants and the available computational environment has limited RAM.

The output identifies candidate SNPs that are present in the African reference panel.

In [28]:
# MATCH EUR GWAS CANDIDATE SNPs TO THE AFRICAN LD REFERENCE PVAR

matched_snps = []

with open(AFR_PVAR, "r", encoding="utf-8") as f:

    # Skip PVAR metadata lines and locate header
    while True:
        line = f.readline()

        if not line:
            raise ValueError(
                "Reached end of PVAR before finding header."
            )

        if not line.startswith("##"):
            header = line.rstrip("\n").split("\t")
            break

    print("PVAR header:")
    print(header)

    # Locate variant ID column
    if "ID" in header:
        id_idx = header.index("ID")
    elif "#ID" in header:
        id_idx = header.index("#ID")
    else:
        raise ValueError(
            f"Could not find variant ID column in PVAR header: {header}"
        )

    print(f"\nVariant ID column index: {id_idx}")
    print(f"Variant ID column name: {header[id_idx]}")

    # Scan variants without loading the entire PVAR into memory

    for line_number, line in enumerate(f, start=2):

        fields = line.rstrip("\n").split("\t")

        if len(fields) <= id_idx:
            continue

        snp = fields[id_idx]

        if snp in candidate_snps:
            matched_snps.append(snp)

        if line_number % 5_000_000 == 0:
            print(
                f"Scanned {line_number:,} PVAR rows | "
                f"Matched {len(matched_snps):,}"
            )

print("\nPVAR scan complete.")

# Remove any duplicate matches

matched_snps = sorted(set(matched_snps))

print(
    f"Candidate EUR GWAS SNPs found in AFR reference: "
    f"{len(matched_snps):,}"
)

# Save matched candidate SNP list

MATCHED_SNP_LIST = (
    CANDIDATE_DIR /
    "MVP_T2D_Phe_250_2_EUR_candidate_SNPs_in_1000G_AFR.txt"
)

pd.Series(
    matched_snps,
    dtype="string"
).to_csv(
    MATCHED_SNP_LIST,
    index=False,
    header=False
)

print("Matched SNP list saved:")
print(MATCHED_SNP_LIST)

print(
    f"Unique matched SNPs: "
    f"{len(matched_snps):,}"
)

PVAR header:
['#CHROM', 'POS', 'ID', 'REF', 'ALT']

Variant ID column index: 2
Variant ID column name: ID
Scanned 5,000,000 PVAR rows | Matched 176,881
Scanned 10,000,000 PVAR rows | Matched 360,330
Scanned 15,000,000 PVAR rows | Matched 546,339
Scanned 20,000,000 PVAR rows | Matched 740,684
Scanned 25,000,000 PVAR rows | Matched 925,405
Scanned 30,000,000 PVAR rows | Matched 1,111,907
Scanned 35,000,000 PVAR rows | Matched 1,324,611
Scanned 40,000,000 PVAR rows | Matched 1,509,262
Scanned 45,000,000 PVAR rows | Matched 1,694,554
Scanned 50,000,000 PVAR rows | Matched 1,878,155
Scanned 55,000,000 PVAR rows | Matched 2,073,535
Scanned 60,000,000 PVAR rows | Matched 2,258,780
Scanned 65,000,000 PVAR rows | Matched 2,436,850
Scanned 70,000,000 PVAR rows | Matched 2,610,331
Scanned 75,000,000 PVAR rows | Matched 2,797,383
Scanned 80,000,000 PVAR rows | Matched 2,962,940

PVAR scan complete.
Candidate EUR GWAS SNPs found in AFR reference: 3,008,384
Matched SNP list saved:
/content/drive/MyD

## 20. Create chromosome-specific candidate SNP extraction lists

The matched candidate SNPs are separated by chromosome.

A separate extraction list is created for each chromosome so that chromosome-specific African reference datasets can be constructed.

This chromosome-wise strategy reduces memory requirements during downstream PLINK processing.

In [30]:
from pathlib import Path
import pandas as pd

# 1. PATHS

REFERENCE_DIR = Path(
    "/content/drive/MyDrive/PRS_T2D_E-A/reference"
)

CANDIDATE_DIR = Path(
    "/content/drive/MyDrive/PRS_T2D_E-A/results/candidate_snps"
)

MATCHED_SNP_LIST = (
    CANDIDATE_DIR /
    "MVP_T2D_Phe_250_2_EUR_candidate_SNPs_in_1000G_AFR.txt"
)

CHUNK_DIR = (
    REFERENCE_DIR /
    "EA_candidate_chunks"
)

CHUNK_DIR.mkdir(
    parents=True,
    exist_ok=True
)

if not MATCHED_SNP_LIST.exists():
    raise FileNotFoundError(
        f"Matched SNP list not found:\n{MATCHED_SNP_LIST}"
    )

# 2. READ MATCHED SNPs

matched = pd.read_csv(
    MATCHED_SNP_LIST,
    header=None,
    names=["SNP"],
    dtype="string"
)

matched = matched.dropna()

matched["SNP"] = (
    matched["SNP"]
    .str.strip()
)

matched = matched.drop_duplicates()

print(
    f"Matched SNPs: {len(matched):,}"
)

# 3. SCAN AFRICAN PVAR FOR CHROMOSOME INFORMATION

matched_set = set(
    matched["SNP"]
)

chromosome_snps = {}

AFR_PVAR = Path(
    str(REFERENCE_DIR / "AFR_only") + ".pvar"
)

if not AFR_PVAR.exists():
    raise FileNotFoundError(
        f"African reference PVAR not found:\n{AFR_PVAR}"
    )

print("\nScanning AFR PVAR...")

with open(
    AFR_PVAR,
    "r",
    encoding="utf-8"
) as f:

    # Skip ## metadata lines

    while True:

        line = f.readline()

        if not line:
            raise ValueError(
                "Could not find PVAR header."
            )

        if not line.startswith("##"):
            header = (
                line
                .rstrip("\n")
                .split("\t")
            )
            break

    chrom_idx = header.index("#CHROM")
    id_idx = header.index("ID")

    print(
        f"Chromosome column index: {chrom_idx}"
    )

    print(
        f"Variant ID column index: {id_idx}"
    )

    # Scan variants without loading
    # the entire PVAR into memory

    for line_number, line in enumerate(
        f,
        start=1
    ):

        fields = (
            line
            .rstrip("\n")
            .split("\t")
        )

        if len(fields) <= max(
            chrom_idx,
            id_idx
        ):
            continue

        chrom = fields[chrom_idx]
        snp = fields[id_idx]

        if snp in matched_set:

            chromosome_snps.setdefault(
                chrom,
                []
            ).append(snp)

        if line_number % 5_000_000 == 0:

            print(
                f"Scanned "
                f"{line_number:,} variants"
            )

print("\nPVAR scan complete.")

# 4. SAVE ONE EXTRACTION LIST PER CHROMOSOME

for chrom, snps in sorted(
    chromosome_snps.items(),
    key=lambda x: str(x[0])
):

    output = (
        CHUNK_DIR /
        f"chr{chrom}_extract.txt"
    )

    unique_snps = sorted(
        set(snps)
    )

    pd.Series(
        unique_snps,
        dtype="string"
    ).to_csv(
        output,
        index=False,
        header=False
    )

    print(
        f"chr{chrom}: "
        f"{len(unique_snps):,} SNPs -> "
        f"{output.name}"
    )

print(
    "\nChromosome extraction lists created."
)

print(
    f"Directory: {CHUNK_DIR}"
)

Matched SNPs: 3,008,384

Scanning AFR PVAR...
Chromosome column index: 0
Variant ID column index: 2
Scanned 5,000,000 variants
Scanned 10,000,000 variants
Scanned 15,000,000 variants
Scanned 20,000,000 variants
Scanned 25,000,000 variants
Scanned 30,000,000 variants
Scanned 35,000,000 variants
Scanned 40,000,000 variants
Scanned 45,000,000 variants
Scanned 50,000,000 variants
Scanned 55,000,000 variants
Scanned 60,000,000 variants
Scanned 65,000,000 variants
Scanned 70,000,000 variants
Scanned 75,000,000 variants
Scanned 80,000,000 variants

PVAR scan complete.
chr1: 233,846 SNPs -> chr1_extract.txt
chr10: 149,668 SNPs -> chr10_extract.txt
chr11: 158,751 SNPs -> chr11_extract.txt
chr12: 141,381 SNPs -> chr12_extract.txt
chr13: 103,386 SNPs -> chr13_extract.txt
chr14: 90,371 SNPs -> chr14_extract.txt
chr15: 88,561 SNPs -> chr15_extract.txt
chr16: 96,715 SNPs -> chr16_extract.txt
chr17: 84,845 SNPs -> chr17_extract.txt
chr18: 82,126 SNPs -> chr18_extract.txt
chr19: 61,597 SNPs -> chr19_e

## 21. Construct chromosome-specific candidate-restricted African LD references

Each chromosome-specific extraction list is applied to the African 1000 Genomes reference.

This produces candidate-restricted LD reference datasets containing only the variants required for downstream LD clumping.

Processing is performed chromosome by chromosome to reduce memory requirements.

The resulting reference datasets are stored as persistent checkpoint files.

In [6]:
import subprocess
import os

# 1. PATHS

base = "/content/drive/MyDrive/PRS_T2D_E-A/"

plink2 = "/content/plink2/plink2"

# Existing African-only reference
afr_reference = base + "reference/AFR_only"

# Chromosome-specific candidate SNP lists
extract_dir = (
    base +
    "reference/EA_candidate_chunks/"
)

# Persistent chromosome-specific LD references
chunk_dir = (
    base +
    "results/ld_clumping/reference_chunks/"
)

os.makedirs(
    chunk_dir,
    exist_ok=True
)

print(
    "Creating chromosome-specific AFR "
    "LD reference datasets...\n"
)

# 2. PROCESS CHROMOSOMES

for chrom in range(1, 23):

    out_prefix = (
        chunk_dir +
        f"AFR_candidate_chr{chrom}_nodup"
    )

    extract_file = (
        extract_dir +
        f"chr{chrom}_extract.txt"
    )

    # Check whether extract list exists

    if not os.path.exists(extract_file):

        print(
            f"SKIP chr{chrom}: "
            f"extract file not found"
        )

        continue

    # Skip if already successfully created

    required = [
        out_prefix + ".pgen",
        out_prefix + ".pvar",
        out_prefix + ".psam"
    ]

    if all(
        os.path.exists(x)
        for x in required
    ):

        print(
            f"SKIP chr{chrom}: "
            f"reference already exists"
        )

        continue

    # PLINK2 command

    cmd = [
        plink2,
        "--pfile",
        afr_reference,
        "--chr",
        str(chrom),
        "--extract",
        extract_file,
        "--rm-dup",
        "force-first",
        "--make-pgen",
        "--memory",
        "9000",
        "--out",
        out_prefix
    ]

    print(
        f"Processing chromosome {chrom}..."
    )

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )

    if result.returncode == 0:

        print(
            f"OK chr{chrom}"
        )

    else:

        print(
            f"FAILED chr{chrom} "
            f"— exit code {result.returncode}"
        )

        print(
            result.stderr[-1000:]
        )

print(
    "\nChromosome-specific AFR reference "
    "construction finished."
)

Creating chromosome-specific AFR LD reference datasets...

Processing chromosome 1...
OK chr1
Processing chromosome 2...
OK chr2
Processing chromosome 3...
OK chr3
Processing chromosome 4...
OK chr4
Processing chromosome 5...
OK chr5
Processing chromosome 6...
OK chr6
Processing chromosome 7...
OK chr7
Processing chromosome 8...
OK chr8
Processing chromosome 9...
OK chr9
Processing chromosome 10...
OK chr10
Processing chromosome 11...
OK chr11
Processing chromosome 12...
OK chr12
Processing chromosome 13...
OK chr13
Processing chromosome 14...
OK chr14
Processing chromosome 15...
OK chr15
Processing chromosome 16...
OK chr16
Processing chromosome 17...
OK chr17
Processing chromosome 18...
OK chr18
Processing chromosome 19...
OK chr19
Processing chromosome 20...
OK chr20
Processing chromosome 21...
OK chr21
Processing chromosome 22...
OK chr22

Chromosome-specific AFR reference construction finished.


## 22. Verify chromosome-specific African reference datasets

Each chromosome-specific reference dataset is checked for the expected PLINK file components.

This verification ensures that all chromosome-specific references required for LD clumping were successfully generated before computationally expensive clumping begins.

In [7]:
import os

chunk_dir = "/content/drive/MyDrive/PRS_T2D_E-A/results/ld_clumping/reference_chunks/"

print("Chromosome-specific reference verification\n")

total_ok = 0

for chrom in range(1, 23):

    prefix = chunk_dir + f"AFR_candidate_chr{chrom}_nodup"

    pgen = prefix + ".pgen"
    pvar = prefix + ".pvar"
    psam = prefix + ".psam"

    if os.path.exists(pgen) and os.path.exists(pvar) and os.path.exists(psam):

        with open(pvar, "r") as f:
            n_var = sum(
                1 for line in f
                if not line.startswith("#")
            )

        print(f"OK chr{chrom}: {n_var:,} variants")
        total_ok += 1

    else:

        missing = []

        if not os.path.exists(pgen):
            missing.append(".pgen")

        if not os.path.exists(pvar):
            missing.append(".pvar")

        if not os.path.exists(psam):
            missing.append(".psam")

        print(f"FAILED chr{chrom}: missing {missing}")

print(f"\n{total_ok}/22 chromosomes complete.")

Chromosome-specific reference verification

OK chr1: 233,846 variants
OK chr2: 259,535 variants
OK chr3: 225,394 variants
OK chr4: 211,555 variants
OK chr5: 193,355 variants
OK chr6: 217,502 variants
OK chr7: 172,940 variants
OK chr8: 171,242 variants
OK chr9: 130,666 variants
OK chr10: 149,668 variants
OK chr11: 158,751 variants
OK chr12: 141,381 variants
OK chr13: 103,386 variants
OK chr14: 90,371 variants
OK chr15: 88,561 variants
OK chr16: 96,715 variants
OK chr17: 84,845 variants
OK chr18: 82,126 variants
OK chr19: 61,597 variants
OK chr20: 63,745 variants
OK chr21: 32,281 variants
OK chr22: 38,922 variants

22/22 chromosomes complete.


## 23. Preliminary chromosome-wise LD clumping

A preliminary chromosome-wise LD clumping run is performed using the full harmonized GWAS and `P1 = 1`.

This stage serves as a computational/reference check of the chromosome-specific clumping infrastructure.

The outputs are stored separately from the threshold-specific clumping results and are not used to construct the final PRS weights.

In [8]:
import subprocess
import os

base = "/content/drive/MyDrive/PRS_T2D_E-A/"
plink2 = "/content/plink2/plink2"

# Chromosome-specific African LD reference datasets
chunk_dir = base + "results/ld_clumping/reference_chunks/"

# European-ancestry harmonized GWAS
harm = base + "results/harmonized/MVP_T2D_Phe_250_2_EUR_harmonized.tsv.gz"

# Preliminary clumping output directory
clump_dir = base + "results/ld_clumping/normal_clumps/"

os.makedirs(clump_dir, exist_ok=True)

print("Starting chromosome-wise LD clumping...\n")

for chrom in range(1, 23):

    reference_prefix = (
        chunk_dir + f"AFR_candidate_chr{chrom}_nodup"
    )

    clump_prefix = (
        clump_dir + f"clump_chr{chrom}_nodup"
    )

    # Required reference files
    required = [
        reference_prefix + ".pgen",
        reference_prefix + ".pvar",
        reference_prefix + ".psam"
    ]

    if not all(os.path.exists(x) for x in required):
        print(f"FAILED chr{chrom}: reference dataset missing")
        continue

    cmd = [
        plink2,
        "--pfile", reference_prefix,
        "--clump", harm,
        "--clump-p1", "1",
        "--clump-r2", "0.1",
        "--clump-kb", "250",
        "--memory", "7000",
        "--out", clump_prefix
    ]

    print(f"Running LD clumping for chromosome {chrom}...")

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )

    clumps_file = clump_prefix + ".clumps"

    if result.returncode == 0 and os.path.exists(clumps_file):

        size_kb = os.path.getsize(clumps_file) / 1024

        print(
            f"OK chr{chrom}: "
            f"{os.path.basename(clumps_file)} "
            f"({size_kb:.1f} KB)"
        )

    else:

        print(
            f"FAILED chr{chrom}: "
            f"PLINK exit code {result.returncode}"
        )

        if result.stderr:
            print(result.stderr[-1500:])

print("\nLD clumping loop finished.")

Starting chromosome-wise LD clumping...

Running LD clumping for chromosome 1...
OK chr1: clump_chr1_nodup.clumps (4199.1 KB)
Running LD clumping for chromosome 2...
OK chr2: clump_chr2_nodup.clumps (4657.7 KB)
Running LD clumping for chromosome 3...
OK chr3: clump_chr3_nodup.clumps (3957.7 KB)
Running LD clumping for chromosome 4...
OK chr4: clump_chr4_nodup.clumps (3543.8 KB)
Running LD clumping for chromosome 5...
OK chr5: clump_chr5_nodup.clumps (3427.9 KB)
Running LD clumping for chromosome 6...
OK chr6: clump_chr6_nodup.clumps (3607.1 KB)
Running LD clumping for chromosome 7...
OK chr7: clump_chr7_nodup.clumps (3012.4 KB)
Running LD clumping for chromosome 8...
OK chr8: clump_chr8_nodup.clumps (2928.1 KB)
Running LD clumping for chromosome 9...
OK chr9: clump_chr9_nodup.clumps (2341.9 KB)
Running LD clumping for chromosome 10...
OK chr10: clump_chr10_nodup.clumps (2644.4 KB)
Running LD clumping for chromosome 11...
OK chr11: clump_chr11_nodup.clumps (2797.8 KB)
Running LD clumpin

In [9]:
import os

clump_dir = "/content/drive/MyDrive/PRS_T2D_E-A/results/ld_clumping/normal_clumps/"

print("PRELIMINARY CLUMPING CHECK\n")

completed = []

for chrom in range(1, 23):

    prefix = clump_dir + f"clump_chr{chrom}_nodup"
    clumps_file = prefix + ".clumps"

    if os.path.exists(clumps_file):
        size_kb = os.path.getsize(clumps_file) / 1024
        print(f"OK chr{chrom}: {size_kb:.1f} KB")
        completed.append(chrom)
    else:
        print(f"MISSING chr{chrom}")

print(
    f"\nCompleted: {len(completed)}/22 chromosomes"
)

if len(completed) == 22:
    print("ALL 22 CHROMOSOMES COMPLETED.")
else:
    missing = [c for c in range(1, 23) if c not in completed]
    print("Missing:", missing)

PRELIMINARY CLUMPING CHECK

OK chr1: 4199.1 KB
OK chr2: 4657.7 KB
OK chr3: 3957.7 KB
OK chr4: 3543.8 KB
OK chr5: 3427.9 KB
OK chr6: 3607.1 KB
OK chr7: 3012.4 KB
OK chr8: 2928.1 KB
OK chr9: 2341.9 KB
OK chr10: 2644.4 KB
OK chr11: 2797.8 KB
OK chr12: 2563.7 KB
OK chr13: 1875.7 KB
OK chr14: 1656.9 KB
OK chr15: 1620.5 KB
OK chr16: 1806.7 KB
OK chr17: 1571.0 KB
OK chr18: 1487.8 KB
OK chr19: 1164.1 KB
OK chr20: 1156.8 KB
OK chr21: 594.7 KB
OK chr22: 702.6 KB

Completed: 22/22 chromosomes
ALL 22 CHROMOSOMES COMPLETED.


## 24. Threshold-specific chromosome-wise LD clumping

LD clumping is performed separately for each of the six GWAS P-value thresholds.

For each threshold:

- the corresponding candidate SNP set is used;
- chromosome-specific African reference data are used;
- the index-variant significance threshold is set to the corresponding GWAS P-value threshold;
- the LD threshold is set to r² = 0.1;
- the physical distance window is set to 250 kb; and
- secondary clumping variants are allowed up to P = 1.0.

Chromosome-wise processing reduces memory requirements and allows failed jobs to be identified and rerun independently.

There are:

6 thresholds × 22 autosomes = 132 autosomal clumping jobs.

If chromosome X is intentionally included, it should be documented and processed separately rather than silently mixing it into the 132-job count.

In [10]:
import os
import subprocess
from pathlib import Path

PLINK = "/content/plink2/plink2"

base = Path("/content/drive/MyDrive/PRS_T2D_E-A/")

reference_dir = (
    base / "results/ld_clumping/reference_chunks"
)

input_dir = (
    base / "results/ld_clumping"
)

output_dir = (
    base / "results/ld_clumping/threshold_clumps"
)

thresholds = {
    "P5e-08": 5e-8,
    "P1e-05": 1e-5,
    "P0.0001": 1e-4,
    "P0.001": 1e-3,
    "P0.01": 1e-2,
    "P0.05": 5e-2
}

output_dir.mkdir(parents=True, exist_ok=True)

success = []
failed = []
skipped = []

total_jobs = len(thresholds) * 22
completed_jobs = 0

print("=" * 70)
print("T2D PRS LD CLUMPING: EUR GWAS + AFR LD")
print("=" * 70)
print(f"Total combinations: {total_jobs}")
print("Thresholds: 6")
print("Chromosomes: 22")
print("Memory per PLINK process: 9000 MB")
print("=" * 70)

for threshold_name, threshold in thresholds.items():

    print("\n" + "=" * 70)
    print(
        f"THRESHOLD: {threshold_name} "
        f"(P <= {threshold})"
    )
    print("=" * 70)

    clump_input = (
        input_dir /
        f"MVP_T2D_Phe_250_2_EUR_{threshold_name}_clump_input.txt"
    )

    if not clump_input.exists():

        print(
            f"ERROR: Input file not found:\n"
            f"{clump_input}"
        )

        for chrom in range(1, 23):
            failed.append(
                (threshold_name, chrom, "missing_input")
            )

        continue

    threshold_out = (
        output_dir / threshold_name
    )

    threshold_out.mkdir(
        parents=True,
        exist_ok=True
    )

    for chrom in range(1, 23):

        reference_prefix = (
            reference_dir /
            f"AFR_candidate_chr{chrom}_nodup"
        )

        output_prefix = (
            threshold_out /
            f"clump_chr{chrom}"
        )

        clumps_file = Path(
            str(output_prefix) + ".clumps"
        )

        completed_jobs += 1

        print("\n" + "-" * 70)
        print(
            f"[{completed_jobs}/{total_jobs}] "
            f"{threshold_name} | chr{chrom}"
        )

        # ----------------------------------------------------
        # CHECK AFRICAN LD REFERENCE
        # ----------------------------------------------------

        required_reference_files = [
            Path(str(reference_prefix) + ".pgen"),
            Path(str(reference_prefix) + ".pvar"),
            Path(str(reference_prefix) + ".psam")
        ]

        missing_reference = [
            str(f)
            for f in required_reference_files
            if not f.exists()
        ]

        if missing_reference:

            print("FAILED: Missing reference file(s)")

            for f in missing_reference:
                print(f"  {f}")

            failed.append(
                (threshold_name, chrom, "missing_reference")
            )

            continue

        # ----------------------------------------------------
        # SKIP EXISTING SUCCESSFUL OUTPUT
        # ----------------------------------------------------

        if clumps_file.exists():

            print(
                "SKIPPED: .clumps already exists"
            )

            print(
                f"Existing file: {clumps_file}"
            )

            skipped.append(
                (threshold_name, chrom)
            )

            continue

        # ----------------------------------------------------
        # PLINK COMMAND
        # ----------------------------------------------------

        cmd = [
            PLINK,

            "--pfile",
            str(reference_prefix),

            "--clump",
            str(clump_input),

            "--clump-p1",
            str(threshold),

            "--clump-p2",
            "1",

            "--clump-r2",
            "0.1",

            "--clump-kb",
            "250",

            "--memory",
            "9000",

            "--out",
            str(output_prefix)
        ]

        print("Running PLINK...")

        try:

            result = subprocess.run(
                cmd,
                capture_output=True,
                text=True,
                timeout=1800
            )

        except subprocess.TimeoutExpired:

            print(
                "FAILED: PLINK timed out after "
                "30 minutes."
            )

            failed.append(
                (threshold_name, chrom, "timeout")
            )

            continue

        except Exception as e:

            print(
                "FAILED: Python could not run PLINK."
            )

            print(repr(e))

            failed.append(
                (threshold_name, chrom, "python_error")
            )

            continue

        # ----------------------------------------------------
        # DETERMINE SUCCESS
        # ----------------------------------------------------

        if clumps_file.exists():

            print(
                f"SUCCESS: {threshold_name} "
                f"chr{chrom}"
            )

            success.append(
                (threshold_name, chrom)
            )

        else:

            print(
                f"FAILED: {threshold_name} "
                f"chr{chrom}"
            )

            print(
                f"PLINK return code: "
                f"{result.returncode}"
            )

            if result.stderr:
                print("\nPLINK STDERR:")
                print(result.stderr[-2500:])

            if result.stdout:
                print("\nPLINK STDOUT:")
                print(result.stdout[-2500:])

            failed.append(
                (
                    threshold_name,
                    chrom,
                    f"return_code_{result.returncode}"
                )
            )


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("CLUMPING SUMMARY")
print("=" * 70)

print(
    f"Successful new runs:       {len(success)}"
)

print(
    f"Skipped existing outputs:  {len(skipped)}"
)

print(
    f"Failed runs:               {len(failed)}"
)

print(
    f"Accounted for:             "
    f"{len(success) + len(skipped) + len(failed)}/{total_jobs}"
)

if skipped:

    print("\nSkipped:")
    for threshold_name, chrom in skipped:
        print(
            f"  {threshold_name} - chr{chrom}"
        )

if failed:

    print("\nFAILED:")
    for threshold_name, chrom, reason in failed:
        print(
            f"  {threshold_name} - chr{chrom} "
            f"[{reason}]"
        )

else:

    print("\nALL 132 CLUMPING COMBINATIONS ACCOUNTED FOR.")

print("\nDone.")

T2D PRS LD CLUMPING: EUR GWAS + AFR LD
Total combinations: 132
Thresholds: 6
Chromosomes: 22
Memory per PLINK process: 9000 MB

THRESHOLD: P5e-08 (P <= 5e-08)

----------------------------------------------------------------------
[1/132] P5e-08 | chr1
Running PLINK...
SUCCESS: P5e-08 chr1

----------------------------------------------------------------------
[2/132] P5e-08 | chr2
Running PLINK...
SUCCESS: P5e-08 chr2

----------------------------------------------------------------------
[3/132] P5e-08 | chr3
Running PLINK...
SUCCESS: P5e-08 chr3

----------------------------------------------------------------------
[4/132] P5e-08 | chr4
Running PLINK...
SUCCESS: P5e-08 chr4

----------------------------------------------------------------------
[5/132] P5e-08 | chr5
Running PLINK...
SUCCESS: P5e-08 chr5

----------------------------------------------------------------------
[6/132] P5e-08 | chr6
Running PLINK...
SUCCESS: P5e-08 chr6

------------------------------------------------

*Note: P5e-08 chromosome 21 produced no clumping results because the T2D GWAS contained no chromosome 21 variants meeting the genome-wide significance threshold of P ≤ 5 × 10⁻⁸. Although 273,556 variants were present on chromosome 21 in the harmonized GWAS, the smallest observed P-value was 7.018 × 10⁻⁷. Therefore, no chromosome 21 SNPs entered the P5e-08 candidate set, leaving nothing to clump. This is an expected result of the T2D GWAS and does not indicate a missing chromosome or an error in the clumping pipeline.*

## 25. Verify threshold-specific LD clumping

The outputs from all threshold-specific chromosome-wise clumping jobs are checked.

The verification confirms:

- the expected number of jobs completed;
- expected `.clumps` files exist;
- files are non-empty; and
- each threshold has results across the expected chromosomes.

A summary table is generated to document the computational completion status.

In [11]:
import os
import pandas as pd

base = "/content/drive/MyDrive/PRS_T2D_E-A/"
clump_base = base + "results/ld_clumping/threshold_clumps/"

thresholds = [
    "P5e-08",
    "P1e-05",
    "P0.0001",
    "P0.001",
    "P0.01",
    "P0.05"
]

print("THRESHOLD-SPECIFIC CLUMPING AUDIT")
print("=" * 70)

summary = []

for threshold in thresholds:

    successful = 0
    empty = 0
    missing = 0
    total_snps = 0

    for chrom in range(1, 23):

        file = (
            clump_base
            + threshold
            + f"/clump_chr{chrom}.clumps"
        )

        if not os.path.exists(file):
            missing += 1
            continue

        if os.path.getsize(file) == 0:
            empty += 1
            continue

        try:
            df = pd.read_csv(
                file,
                sep=r"\s+"
            )

            n = len(df)

            if n > 0:
                successful += 1
                total_snps += n
            else:
                empty += 1

        except Exception as e:

            print(
                f"ERROR: "
                f"{threshold} chr{chrom}: {e}"
            )

    summary.append({
        "Threshold": threshold,
        "Successful chromosomes": successful,
        "Empty files": empty,
        "Missing files": missing,
        "Total clumped SNPs": total_snps
    })

summary_df = pd.DataFrame(summary)

print(summary_df.to_string(index=False))

THRESHOLD-SPECIFIC CLUMPING AUDIT
Threshold  Successful chromosomes  Empty files  Missing files  Total clumped SNPs
   P5e-08                      22            0              0                4944
   P1e-05                      22            0              0               16558
  P0.0001                      22            0              0               37511
   P0.001                      22            0              0              107335
    P0.01                      22            0              0              364262
    P0.05                      22            0              0              920240


## 26. Construct PRS weight files

The index variants identified by LD clumping are extracted from the harmonized GWAS.

For each P-value threshold:

1. the index SNP identifiers are collected from the chromosome-specific `.clumps` files;
2. duplicate index SNPs are removed;
3. the SNP identifiers are matched back to the harmonized GWAS;
4. the corresponding effect allele, other allele, effect estimate, standard error, allele frequency and P-value are retained; and
5. the resulting variant table is saved as the PRS weight file.

The GWAS `BETA` value is used as the PRS weight.

The effect allele is the allele corresponding to the reported GWAS effect estimate.

### PRS calculation principle

For an individual *i*, the polygenic risk score is conceptually calculated as:

PRSᵢ = Σⱼ Gᵢⱼ × βⱼ

where:

- Gᵢⱼ is the dosage of the effect allele for variant *j* in individual *i*; and
- βⱼ is the GWAS effect estimate for variant *j*.

Actual individual-level PRS scoring is not performed in this notebook because an independent target genotype dataset is not yet available.

In [12]:
import os
import pandas as pd

base = "/content/drive/MyDrive/PRS_T2D_E-A/"

harmonized_file = (
    base +
    "results/harmonized/MVP_T2D_Phe_250_2_EUR_harmonized.tsv.gz"
)

clump_base = (
    base +
    "results/ld_clumping/threshold_clumps/"
)

weight_base = (
    base +
    "results/weights/"
)

thresholds = [
    "P5e-08",
    "P1e-05",
    "P0.0001",
    "P0.001",
    "P0.01",
    "P0.05"
]

# 1. Load harmonized EUR GWAS

print("Loading harmonized EUR GWAS...")

gwas = pd.read_csv(
    harmonized_file,
    sep="\t",
    compression="gzip"
)

print("GWAS variants:", len(gwas))
print("GWAS columns:", gwas.columns.tolist())

# Keep fields required for PRS weights
gwas_weights = gwas[
    ["SNP", "CHR", "BP", "A1", "A2", "EAF", "BETA", "SE", "P", "N"]
].copy()

gwas_weights["SNP"] = gwas_weights["SNP"].astype(str)

# Remove duplicate SNP IDs
gwas_weights = gwas_weights.drop_duplicates(
    subset="SNP",
    keep="first"
)

# Index by SNP for fast matching
gwas_weights = gwas_weights.set_index("SNP")

print("Unique GWAS SNPs:", len(gwas_weights))


# 2. Process each P-value threshold

for threshold in thresholds:

    print(f"\nPROCESSING {threshold}")

    threshold_dir = (
        clump_base +
        threshold +
        "/"
    )

    all_clumped = []

    # Read chromosome-specific clump files

    for chrom in range(1, 23):

        clump_file = (
            threshold_dir +
            f"clump_chr{chrom}.clumps"
        )

        if not os.path.exists(clump_file):

            print(
                f"WARNING: missing chr{chrom} clump file"
            )

            continue

        clumps = pd.read_csv(
            clump_file,
            sep=r"\s+"
        )

        if "ID" not in clumps.columns:

            print(
                f"WARNING: ID column missing in chr{chrom}"
            )

            continue

        ids = (
            clumps["ID"]
            .dropna()
            .astype(str)
        )

        all_clumped.extend(ids.tolist())

    # Combine and remove duplicate index SNPs

    clumped_snps = (
        pd.Series(
            all_clumped,
            dtype="string"
        )
        .drop_duplicates()
    )

    print(
        "Unique clumped SNPs:",
        len(clumped_snps)
    )

    # Match index SNPs to the EUR harmonized GWAS

    matched = gwas_weights[
        gwas_weights.index.isin(clumped_snps)
    ].copy()

    print(
        "Matched to harmonized GWAS:",
        len(matched)
    )

    # Matching rate

    if len(clumped_snps) > 0:

        match_rate = (
            len(matched) /
            len(clumped_snps)
        ) * 100

        print(
            f"Matching rate: {match_rate:.2f}%"
        )

    # Restore SNP as a column

    matched = matched.reset_index()

    # Create threshold-specific output directory

    output_dir = (
        weight_base +
        threshold +
        "/"
    )

    os.makedirs(
        output_dir,
        exist_ok=True
    )

    output_file = (
        output_dir +
        "MVP_T2D_EUR_weights.tsv"
    )

    matched.to_csv(
        output_file,
        sep="\t",
        index=False
    )

    print("Saved:")
    print(output_file)

print("\nPRS WEIGHT CONSTRUCTION COMPLETE")

Loading harmonized EUR GWAS...
GWAS variants: 18690053
GWAS columns: ['SNP', 'CHR', 'BP', 'A1', 'A2', 'EAF', 'BETA', 'SE', 'P', 'N']
Unique GWAS SNPs: 18645319

PROCESSING P5e-08
Unique clumped SNPs: 4944
Matched to harmonized GWAS: 4944
Matching rate: 100.00%
Saved:
/content/drive/MyDrive/PRS_T2D_E-A/results/weights/P5e-08/MVP_T2D_EUR_weights.tsv

PROCESSING P1e-05
Unique clumped SNPs: 16558
Matched to harmonized GWAS: 16558
Matching rate: 100.00%
Saved:
/content/drive/MyDrive/PRS_T2D_E-A/results/weights/P1e-05/MVP_T2D_EUR_weights.tsv

PROCESSING P0.0001
Unique clumped SNPs: 37511
Matched to harmonized GWAS: 37511
Matching rate: 100.00%
Saved:
/content/drive/MyDrive/PRS_T2D_E-A/results/weights/P0.0001/MVP_T2D_EUR_weights.tsv

PROCESSING P0.001
Unique clumped SNPs: 107335
Matched to harmonized GWAS: 107335
Matching rate: 100.00%
Saved:
/content/drive/MyDrive/PRS_T2D_E-A/results/weights/P0.001/MVP_T2D_EUR_weights.tsv

PROCESSING P0.01
Unique clumped SNPs: 364262
Matched to harmonized GW

## 27. Verify the final PRS weight files

The six final PRS weight files are verified.

For each threshold, the following are checked:

- file existence;
- number of variants;
- required columns;
- duplicate SNP identifiers;
- missing effect estimates;
- missing effect alleles; and
- successful matching of clumped index SNPs back to the harmonized GWAS.

This represents the final computational checkpoint of the discovery-stage PRS pipeline.

In [14]:
import os
import pandas as pd

base = "/content/drive/MyDrive/PRS_T2D_E-A/"

weight_base = (
    base +
    "results/weights/"
)

harmonized_file = (
    base +
    "results/harmonized/MVP_T2D_Phe_250_2_EUR_harmonized.tsv.gz"
)

thresholds = [
    "P5e-08",
    "P1e-05",
    "P0.0001",
    "P0.001",
    "P0.01",
    "P0.05"
]

required_columns = [
    "SNP",
    "CHR",
    "BP",
    "A1",
    "A2",
    "EAF",
    "BETA",
    "SE",
    "P",
    "N"
]

# Load harmonized GWAS SNP IDs for matching verification

print("Loading harmonized GWAS SNP IDs...")

harmonized_snps = set()

for chunk in pd.read_csv(
    harmonized_file,
    sep="\t",
    usecols=["SNP"],
    chunksize=500_000
):
    harmonized_snps.update(
        chunk["SNP"].dropna().astype(str)
    )

print(
    "Unique harmonized GWAS SNPs:",
    f"{len(harmonized_snps):,}"
)

print("\nVERIFYING FINAL PRS WEIGHT FILES")

for threshold in thresholds:

    print(f"\n{threshold}")

    weight_file = (
        weight_base +
        threshold +
        "/MVP_T2D_EUR_weights.tsv"
    )

    # 1. File existence

    if not os.path.exists(weight_file):
        print("FAIL: weight file does not exist")
        continue

    print("File exists: PASS")

    # Load weight file

    weights = pd.read_csv(
        weight_file,
        sep="\t"
    )

    # 2. Number of variants

    print(
        "Number of variants:",
        f"{len(weights):,}"
    )

    # 3. Required columns

    missing_columns = [
        col for col in required_columns
        if col not in weights.columns
    ]

    if missing_columns:
        print(
            "Required columns: FAIL",
            missing_columns
        )
    else:
        print("Required columns: PASS")

    # 4. Duplicate SNP identifiers

    duplicate_count = weights["SNP"].duplicated().sum()

    if duplicate_count == 0:
        print("Duplicate SNP IDs: PASS")
    else:
        print(
            "Duplicate SNP IDs: FAIL",
            duplicate_count
        )

    # 5. Missing effect estimates

    missing_beta = weights["BETA"].isna().sum()

    if missing_beta == 0:
        print("Missing BETA values: PASS")
    else:
        print(
            "Missing BETA values: FAIL",
            missing_beta
        )

    # 6. Missing effect alleles

    missing_a1 = weights["A1"].isna().sum()

    if missing_a1 == 0:
        print("Missing effect alleles: PASS")
    else:
        print(
            "Missing effect alleles: FAIL",
            missing_a1
        )

    # 7. Match weight SNPs to harmonized GWAS

    weight_snps = set(
        weights["SNP"]
        .dropna()
        .astype(str)
    )

    matched_snps = weight_snps.intersection(
        harmonized_snps
    )

    unmatched_count = (
        len(weight_snps) -
        len(matched_snps)
    )

    if unmatched_count == 0:
        print(
            "Matching to harmonized GWAS: PASS"
        )
    else:
        print(
            "Matching to harmonized GWAS: FAIL",
            unmatched_count,
            "unmatched SNPs"
        )

    # Matching rate

    if len(weight_snps) > 0:
        match_rate = (
            len(matched_snps) /
            len(weight_snps)
        ) * 100

        print(
            f"Matching rate: {match_rate:.2f}%"
        )

print("\nFINAL PRS WEIGHT VERIFICATION COMPLETE")

Loading harmonized GWAS SNP IDs...
Unique harmonized GWAS SNPs: 18,645,319

VERIFYING FINAL PRS WEIGHT FILES

P5e-08
File exists: PASS
Number of variants: 4,944
Required columns: PASS
Duplicate SNP IDs: PASS
Missing BETA values: PASS
Missing effect alleles: PASS
Matching to harmonized GWAS: PASS
Matching rate: 100.00%

P1e-05
File exists: PASS
Number of variants: 16,558
Required columns: PASS
Duplicate SNP IDs: PASS
Missing BETA values: PASS
Missing effect alleles: PASS
Matching to harmonized GWAS: PASS
Matching rate: 100.00%

P0.0001
File exists: PASS
Number of variants: 37,511
Required columns: PASS
Duplicate SNP IDs: PASS
Missing BETA values: PASS
Missing effect alleles: PASS
Matching to harmonized GWAS: PASS
Matching rate: 100.00%

P0.001
File exists: PASS
Number of variants: 107,335
Required columns: PASS
Duplicate SNP IDs: PASS
Missing BETA values: PASS
Missing effect alleles: PASS
Matching to harmonized GWAS: PASS
Matching rate: 100.00%

P0.01
File exists: PASS
Number of variant

# Final stopping point

The discovery-stage PRS pipeline is complete.

The pipeline has produced verified PRS weight files at six GWAS P-value thresholds after:

1. GWAS inspection;
2. GWAS quality control;
3. GWAS harmonization;
4. candidate SNP selection;
5. construction of an African 1000 Genomes LD reference;
6. candidate SNP matching;
7. chromosome-specific reference construction;
8. chromosome-wise LD clumping; and
9. PRS weight construction.

The pipeline stops here because an independent target genotype and phenotype dataset is required for actual PRS scoring and validation.

No target-level PRS calculation, regression analysis, ROC-AUC analysis, calibration analysis, or predictive validation is performed in this notebook.

The End.